# Dashboard copilot: train and evaluate the LoRA adapter

Run the cells from top to bottom (**Run All**). Before you start:

1. Right panel, **Session options**: Accelerator = **GPU P100** (faster for this job) or **GPU T4 x2**; Internet = **On**. The job uses one GPU only.
2. **Add-ons > Secrets**: add a secret named `HF_TOKEN` (a Hugging Face read-only token) and attach it to this notebook.
3. Your Hugging Face account must have accepted the license of `google/functiongemma-270m-it`.

The notebook never prints the token. Estimated time: 2 to 4 hours (measured 1.1 s per sequence on a GTX 1650; Kaggle GPUs should be 2.5 to 3.5 times faster; not yet measured on Kaggle). Each progress line shows the elapsed seconds. A Kaggle session stops after 9 hours. At the end, download `copilot_adapter.zip` and `report.json` from the **Output** panel.

In [ ]:
!nvidia-smi
import sys, torch
print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available())

In [ ]:
# Kaggle ships torchao 0.10.0. PEFT finds it and refuses to start (it needs a newer one).
# This job does not use torchao, so remove it.
!pip -q uninstall -y torchao
!pip -q install transformers==5.17.0 tokenizers==0.23.2 huggingface-hub==1.33.0 safetensors==0.8.0 peft==0.21.1 accelerate==1.15.0
import importlib.util, transformers, peft
assert importlib.util.find_spec('torchao') is None, 'torchao is still installed: restart the session, then run this cell again'
print('transformers', transformers.__version__, 'peft', peft.__version__)

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download
token = UserSecretsClient().get_secret('HF_TOKEN')
try:
    BASE = snapshot_download('google/functiongemma-270m-it', local_dir='/kaggle/working/base', token=token)
except Exception as error:
    raise SystemExit('Download failed. Open https://huggingface.co/google/functiongemma-270m-it, '
                     'accept the license, check HF_TOKEN, then run this cell again. ' + type(error).__name__)
finally:
    del token
print('base model files:', sorted(os.listdir(BASE)))

## Source files

Each cell writes one file of the repository. Do not edit them here.

In [ ]:
import os
for d in ['tools/voice/dashboard_llm/train']:
    os.makedirs('/kaggle/working/fyp/' + d, exist_ok=True)
open('/kaggle/working/fyp/tools/__init__.py', 'a').close()
open('/kaggle/working/fyp/tools/voice/__init__.py', 'a').close()

In [ ]:
open('/kaggle/working/fyp/tools/voice/dashboard_llm/__init__.py', 'w').close()  # empty file

In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/service.py
from __future__ import annotations

import json
import math
import os
import re
import sys
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from typing import Any, Callable, Optional

MAX_TEXT = 200
MAX_PAGES = 60
MAX_BODY = 32_768
MAX_ALIASES = 8
MAX_SECONDS = 2.0
DEFAULT_PORT = 8765
DEFAULT_MODEL_DIR = r"D:\models\functiongemma-270m-it"
DEFAULT_ADAPTER_DIR = r"D:\models\functiongemma-dashboard-lora"
THRESHOLD = 0.98
BASE_MODEL_THRESHOLD = 0.999
ESCAPE_ID = 52
TOOL_NAME = "navigate_to_page"
NO_MATCH = "no_match"
PREFILL = f"<start_function_call>call:{TOOL_NAME}{{page:<escape>"
_LOCAL_ORIGIN = re.compile(r"^http://(?:localhost|127\.0\.0\.1)(?::\d{1,5})?$")
_KEY = re.compile(r"^[a-z][a-z0-9_]{0,63}$")
_CALL = re.compile(
    rf"call:{TOOL_NAME}\{{page:<escape>([^<>{{}}]{{1,80}})<escape>\}}"
)

Generate = Callable[[str, "list[dict[str, Any]]"], "tuple[str, float]"]


class RequestError(ValueError):
    pass


def origin_allowed(origin: Optional[str]) -> bool:
    return bool(origin) and _LOCAL_ORIGIN.fullmatch(origin) is not None


def validate_request(payload: Any) -> "tuple[str, list[dict[str, Any]]]":
    if not isinstance(payload, dict):
        raise RequestError("body must be an object")
    text = payload.get("text")
    if not isinstance(text, str) or not text.strip():
        raise RequestError("text must be a non-empty string")
    if len(text) > MAX_TEXT:
        raise RequestError("text too long")
    pages = payload.get("pages")
    if not isinstance(pages, list) or not pages or len(pages) > MAX_PAGES:
        raise RequestError("pages must be a non-empty list")
    clean: list[dict[str, Any]] = []
    seen: set[str] = set()
    for page in pages:
        if not isinstance(page, dict):
            raise RequestError("page must be an object")
        key, label = page.get("key"), page.get("label")
        aliases = page.get("aliases", [])
        if not isinstance(key, str) or not _KEY.fullmatch(key) or key in seen:
            raise RequestError("invalid or duplicate page key")
        if not isinstance(label, str) or len(label) > 80 or "<" in label:
            raise RequestError("invalid page label")
        if not isinstance(aliases, list) or not all(
            isinstance(a, str) and len(a) <= 60 and "<" not in a for a in aliases
        ):
            raise RequestError("invalid aliases")
        seen.add(key)
        clean.append({"key": key, "label": label, "aliases": aliases[:MAX_ALIASES]})
    return " ".join(text.split()), clean


def build_tools(pages: "list[dict[str, Any]]") -> "list[dict[str, Any]]":
    described = "; ".join(
        f"{p['key']} = {p['label']}"
        + (f" ({', '.join(p['aliases'])})" if p["aliases"] else "")
        for p in pages
    )
    described += f"; {NO_MATCH} = none of the pages fit the request"
    return [
        {
            "type": "function",
            "function": {
                "name": TOOL_NAME,
                "description": f"Open one dashboard page. Pages: {described}",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "page": {
                            "type": "string",
                            "description": "The page to open",
                            "enum": [p["key"] for p in pages] + [NO_MATCH],
                        }
                    },
                    "required": ["page"],
                },
            },
        }
    ]


def parse_page(output: str, pages: "list[dict[str, Any]]") -> Optional[str]:
    matches = _CALL.findall(output)
    if len(matches) != 1:
        return None
    value = matches[0].strip().lower()
    if any(p["key"] == value for p in pages):
        return value
    hits = {
        p["key"]
        for p in pages
        if value in {p["label"].lower(), *(a.lower() for a in p["aliases"])}
    }
    return hits.pop() if len(hits) == 1 else None


def propose(generate: Generate, payload: Any, threshold: float) -> "dict[str, Any]":
    started = time.perf_counter()
    text, pages = validate_request(payload)
    output, confidence = generate(text, pages)
    key = parse_page(output, pages)
    if confidence < threshold:
        key = None
    return {
        "pageKey": key,
        "confidence": round(confidence if key else 0.0, 4),
        "latencyMs": int((time.perf_counter() - started) * 1000),
    }


def render_prompt(tok: Any, text: str, pages: "list[dict[str, Any]]") -> str:
    messages = [
        {
            "role": "developer",
            "content": "You are a model that can do function calling with the following functions",
        },
        {"role": "user", "content": text},
    ]
    prompt = tok.apply_chat_template(
        messages, tools=build_tools(pages), add_generation_prompt=True, tokenize=False
    )
    return prompt + PREFILL


def default_threshold(adapter: bool, env: "Optional[str]" = None) -> float:
    value = env if env is not None else os.environ.get("DASHBOARD_VOICE_THRESHOLD")
    if value:
        return float(value)
    return THRESHOLD if adapter else BASE_MODEL_THRESHOLD


def resolve_adapter_dir(env: "Optional[str]" = None) -> "Optional[str]":
    path = env if env is not None else os.environ.get("DASHBOARD_VOICE_ADAPTER_DIR", DEFAULT_ADAPTER_DIR)
    return path if path and os.path.isfile(os.path.join(path, "adapter_config.json")) else None


class GemmaGenerator:
    def __init__(
        self, model_dir: str, max_new_tokens: int = 24, adapter_dir: "Optional[str]" = None
    ) -> None:
        self._dir = model_dir
        self.adapter_dir = adapter_dir
        self._max_new = max_new_tokens
        self._model: Any = None
        self._tok: Any = None
        self._torch: Any = None
        self._lock = threading.Lock()

    def _load(self) -> None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self._torch = torch
        self._tok = AutoTokenizer.from_pretrained(self._dir, local_files_only=True)
        if self._tok.convert_tokens_to_ids("<escape>") != ESCAPE_ID:
            raise RuntimeError("unexpected tokenizer: <escape> id changed")
        model = AutoModelForCausalLM.from_pretrained(
            self._dir, local_files_only=True, dtype=torch.float32
        )
        if self.adapter_dir:
            from peft import PeftModel

            model = PeftModel.from_pretrained(model, self.adapter_dir, local_files_only=True)
            model = model.merge_and_unload()
        model = model.eval()
        self._model = model.to("cuda") if torch.cuda.is_available() else model

    def __call__(self, text: str, pages: "list[dict[str, Any]]") -> "tuple[str, float]":
        with self._lock:
            if self._model is None:
                self._load()
            torch, tok, model = self._torch, self._tok, self._model
            inputs = tok(render_prompt(tok, text, pages), return_tensors="pt", add_special_tokens=False)
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            with torch.inference_mode():
                out = model.generate(
                    **inputs,
                    max_new_tokens=self._max_new,
                    max_time=MAX_SECONDS,
                    do_sample=False,
                    output_scores=True,
                    return_dict_in_generate=True,
                    eos_token_id=[ESCAPE_ID, tok.eos_token_id],
                    pad_token_id=tok.pad_token_id,
                )
            new = out.sequences[0][inputs["input_ids"].shape[-1] :]
            logp = 0.0
            for step, token in enumerate(new):
                probs = torch.softmax(out.scores[step][0].float(), dim=-1)
                logp += math.log(max(float(probs[token]), 1e-12))
            decoded = tok.decode(new, skip_special_tokens=False)
            return f"{PREFILL}{decoded}}}", math.exp(logp)


def make_handler(generate: Generate, threshold: float, adapter: bool = False):
    class Handler(BaseHTTPRequestHandler):
        server_version = "DashboardVoiceLLM"

        def log_message(self, *args: Any) -> None:
            return

        def _cors(self) -> None:
            origin = self.headers.get("Origin")
            if origin_allowed(origin):
                self.send_header("Access-Control-Allow-Origin", origin)
                self.send_header("Vary", "Origin")
                self.send_header("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
                self.send_header("Access-Control-Allow-Headers", "Content-Type")

        def _send(self, status: int, body: "dict[str, Any]") -> None:
            data = json.dumps(body).encode()
            self.send_response(status)
            self.send_header("Content-Type", "application/json")
            self.send_header("Content-Length", str(len(data)))
            self._cors()
            self.end_headers()
            self.wfile.write(data)

        def _origin_rejected(self) -> bool:
            origin = self.headers.get("Origin")
            if origin is not None and not origin_allowed(origin):
                self._send(403, {"error": "origin not allowed"})
                return True
            return False

        def do_OPTIONS(self) -> None:
            if self._origin_rejected():
                return
            self.send_response(204)
            self._cors()
            self.end_headers()

        def do_GET(self) -> None:
            if self._origin_rejected():
                return
            if self.path == "/health":
                self._send(200, {"status": "ok", "threshold": threshold, "adapter": adapter})
            else:
                self._send(404, {"error": "not found"})

        def do_POST(self) -> None:
            if self._origin_rejected():
                return
            if self.path != "/propose":
                self._send(404, {"error": "not found"})
                return
            try:
                length = int(self.headers.get("Content-Length", "0"))
            except ValueError:
                length = -1
            if length <= 0 or length > MAX_BODY:
                self._send(413, {"error": "bad body size"})
                return
            try:
                payload = json.loads(self.rfile.read(length))
                self._send(200, propose(generate, payload, threshold))
            except (RequestError, json.JSONDecodeError) as error:
                self._send(400, {"error": str(error)})
            except Exception:
                self._send(500, {"error": "inference failed"})

    return Handler


def serve() -> None:
    port = int(os.environ.get("DASHBOARD_VOICE_PORT", DEFAULT_PORT))
    model_dir = os.environ.get("DASHBOARD_VOICE_MODEL_DIR", DEFAULT_MODEL_DIR)
    generator = GemmaGenerator(model_dir, adapter_dir=resolve_adapter_dir())
    threshold = default_threshold(generator.adapter_dir is not None)
    threading.Thread(
        target=generator,
        args=("warm up", [{"key": "dashboard", "label": "Dashboard", "aliases": []}]),
        daemon=True,
    ).start()
    server = ThreadingHTTPServer(("127.0.0.1", port), make_handler(generator, threshold, generator.adapter_dir is not None))
    print(f"dashboard voice LLM on http://127.0.0.1:{port} (model warming up, adapter={'on' if generator.adapter_dir else 'off'})")
    server.serve_forever()


if __name__ == "__main__":
    sys.exit(serve())


In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/copilot_schema.py
from __future__ import annotations

import re
from dataclasses import dataclass
from typing import Any, Mapping, Optional, Sequence

NAVIGATE = "navigate_to_page"
NO_MATCH = "no_match"
PREFILL = "<start_function_call>call:"
END_CALL = "<end_function_call>"
DEVELOPER_PROMPT = "You are a model that can do function calling with the following functions"

MAX_VALUE = 80
MAX_POSITION = 50
PERIODS = ("today", "yesterday", "this_week", "last_week", "this_month", "last_month")
ORDER_STATUSES = ("submitted", "approved", "rejected")
ITEM_STATUSES = ("in_review", "resolved", "ignored")
WEEKS = ("this_week", "next_week")

ACTION_SPECS: dict[str, dict[str, Any]] = {
    "search_orders": {
        "description": (
            "Find orders. Use the names exactly as the user said them. "
            "Give only the filters the user mentioned."
        ),
        "params": {
            "salesperson": {"type": "string", "description": "Salesperson name as spoken"},
            "retailer": {"type": "string", "description": "Retailer or shop name as spoken"},
            "period": {"type": "string", "description": "Time range", "enum": list(PERIODS)},
            "status": {"type": "string", "description": "Order status", "enum": list(ORDER_STATUSES)},
        },
        "required": [],
    },
    "export_results": {
        "description": "Export the results currently shown, as a CSV file.",
        "params": {},
        "required": [],
    },
    "daily_briefing": {
        "description": "Show the prioritised list of things that need attention today.",
        "params": {},
        "required": [],
    },
    "explain_item": {
        "description": "Explain one item of the briefing. Omit position for 'this one'.",
        "params": {"position": {"type": "integer", "description": "Item number in the list, from 1"}},
        "required": [],
    },
    "set_item_status": {
        "description": "Set the user's own review status on a briefing item. Omit position for 'it'.",
        "params": {
            "position": {"type": "integer", "description": "Item number in the list, from 1"},
            "status": {"type": "string", "description": "New status", "enum": list(ITEM_STATUSES)},
        },
        "required": ["status"],
    },
    "brief_retailer": {
        "description": "Prepare a visit brief for one retailer. Omit retailer for 'this retailer'.",
        "params": {"retailer": {"type": "string", "description": "Retailer or shop name as spoken"}},
        "required": [],
    },
    "route_why": {
        "description": (
            "Explain why one retailer is on a salesperson's route. "
            "Omit a name when the user means the last one discussed."
        ),
        "params": {
            "retailer": {"type": "string", "description": "Retailer or shop name as spoken"},
            "salesperson": {"type": "string", "description": "Salesperson name as spoken"},
        },
        "required": [],
    },
    "salesperson_visits": {
        "description": (
            "Say which retailers a salesperson should visit, from the weekly ranking. "
            "Omit the name when the user means the last salesperson discussed."
        ),
        "params": {
            "salesperson": {"type": "string", "description": "Salesperson name as spoken"},
            "week": {"type": "string", "description": "Week", "enum": list(WEEKS)},
        },
        "required": [],
    },
    "due_overview": {
        "description": "List the retailers that are due a visit or have stopped buying, by salesperson.",
        "params": {},
        "required": [],
    },
    "open_planner": {
        "description": (
            "Open the route planner for one salesperson and one week. "
            "Omit the name when the user means the last salesperson discussed."
        ),
        "params": {
            "salesperson": {"type": "string", "description": "Salesperson name as spoken"},
            "week": {"type": "string", "description": "Week", "enum": list(WEEKS)},
        },
        "required": [],
    },
    "no_route": {
        "description": "Name the salespeople who have no published route for a week, by plan status.",
        "params": {"week": {"type": "string", "description": "Week", "enum": list(WEEKS)}},
        "required": [],
    },
}
ACTION_ORDER = tuple(ACTION_SPECS)
HOSTED_ONLY_ACTIONS = ("route_why", "salesperson_visits", "due_overview", "open_planner", "no_route")
TRAINED_ACTION_ORDER = tuple(a for a in ACTION_ORDER if a not in HOSTED_ONLY_ACTIONS)

_KEY = re.compile(r"^[a-z][a-z0-9_]{0,63}$")
_CALL = re.compile(r"call:\s*([a-z_][a-z0-9_]*)\{([^{}]*)\}" + re.escape(END_CALL))
_ARG = re.compile(r"([a-z_][a-z0-9_]*):(?:<escape>([^<>{}]{1,%d})<escape>|(-?\d{1,3}))" % MAX_VALUE)


class RequestError(ValueError):
    pass


@dataclass(frozen=True)
class Call:
    name: str
    args: Mapping[str, Any]

    def as_dict(self) -> dict[str, Any]:
        return {"name": self.name, "args": dict(self.args)}


def _json_type(kind: str) -> str:
    return kind


def _function(name: str, description: str, params: Mapping[str, Mapping[str, Any]], required: Sequence[str]) -> dict[str, Any]:
    properties: dict[str, Any] = {}
    for key, spec in params.items():
        entry: dict[str, Any] = {"type": _json_type(spec["type"]), "description": spec["description"]}
        if "enum" in spec:
            entry["enum"] = list(spec["enum"])
        properties[key] = entry
    return {
        "type": "function",
        "function": {
            "name": name,
            "description": description,
            "parameters": {"type": "object", "properties": properties, "required": list(required)},
        },
    }


def _navigate_tool(pages: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    described = "; ".join(
        f"{p['key']} = {p['label']}" + (f" ({', '.join(p['aliases'])})" if p["aliases"] else "")
        for p in pages
    )
    return _function(
        NAVIGATE,
        f"Open one dashboard page. Pages: {described}",
        {"page": {"type": "string", "description": "The page to open", "enum": [p["key"] for p in pages]}},
        ["page"],
    )


def build_tools(pages: Sequence[Mapping[str, Any]], actions: Sequence[str]) -> list[dict[str, Any]]:
    tools: list[dict[str, Any]] = []
    if pages:
        tools.append(_navigate_tool(pages))
    for name in ACTION_ORDER:
        if name in actions:
            spec = ACTION_SPECS[name]
            tools.append(_function(name, spec["description"], spec["params"], spec["required"]))
    tools.append(_function(NO_MATCH, "None of the other functions fit the request.", {}, []))
    return tools


def render_prompt(tok: Any, text: str, pages: Sequence[Mapping[str, Any]], actions: Sequence[str]) -> str:
    messages = [
        {"role": "developer", "content": DEVELOPER_PROMPT},
        {"role": "user", "content": text},
    ]
    prompt = tok.apply_chat_template(
        messages, tools=build_tools(pages, actions), add_generation_prompt=True, tokenize=False
    )
    return prompt + PREFILL


def serialize_call(call: Call | Mapping[str, Any]) -> str:
    name = call.name if isinstance(call, Call) else call["name"]
    args = call.args if isinstance(call, Call) else call.get("args", {})
    parts = []
    for key in sorted(args):
        value = args[key]
        parts.append(f"{key}:{value}" if isinstance(value, int) else f"{key}:<escape>{value}<escape>")
    return f"{name}{{{','.join(parts)}}}{END_CALL}"


def _resolve_page(value: str, pages: Sequence[Mapping[str, Any]]) -> Optional[str]:
    needle = value.strip().lower()
    if any(p["key"] == needle for p in pages):
        return needle
    hits = {
        p["key"]
        for p in pages
        if needle in {p["label"].lower(), *(a.lower() for a in p["aliases"])}
    }
    return hits.pop() if len(hits) == 1 else None


def parse_call(output: str, pages: Sequence[Mapping[str, Any]], actions: Sequence[str]) -> Optional[Call]:
    if output.count("call:") != 1:
        return None
    match = _CALL.search(output)
    if match is None:
        return None
    name, body = match.group(1), match.group(2)
    allowed = {NO_MATCH, *(a for a in actions if a in ACTION_SPECS)}
    if pages:
        allowed.add(NAVIGATE)
    if name not in allowed:
        return None

    args: dict[str, Any] = {}
    position = 0
    while position < len(body):
        arg = _ARG.match(body, position)
        if arg is None:
            return None
        key, text, number = arg.group(1), arg.group(2), arg.group(3)
        if key in args:
            return None
        args[key] = text.strip() if text is not None else int(number)
        position = arg.end()
        if position < len(body):
            if body[position] != ",":
                return None
            position += 1
            if position == len(body):
                return None

    if name == NO_MATCH:
        return Call(NO_MATCH, {}) if not args else None
    if name == NAVIGATE:
        if set(args) != {"page"} or not isinstance(args["page"], str):
            return None
        page = _resolve_page(args["page"], pages)
        return Call(NAVIGATE, {"page": page}) if page else None

    spec = ACTION_SPECS[name]
    for key, value in args.items():
        param = spec["params"].get(key)
        if param is None:
            return None
        if param["type"] == "integer":
            if isinstance(value, bool) or not isinstance(value, int) or not 1 <= value <= MAX_POSITION:
                return None
        else:
            if not isinstance(value, str) or not value:
                return None
            if "enum" in param and value not in param["enum"]:
                return None
    if any(key not in args for key in spec["required"]):
        return None
    if name == "search_orders" and not set(args) & {"salesperson", "retailer", "period", "status"}:
        return None
    return Call(name, args)


def validate_request(payload: Any) -> tuple[str, list[dict[str, Any]], list[str]]:
    if not isinstance(payload, dict):
        raise RequestError("body must be an object")
    text = payload.get("text")
    if not isinstance(text, str) or not text.strip():
        raise RequestError("text must be a non-empty string")
    if len(text) > 200:
        raise RequestError("text too long")
    pages = payload.get("pages", [])
    if not isinstance(pages, list) or len(pages) > 60:
        raise RequestError("pages must be a list of at most 60")
    clean: list[dict[str, Any]] = []
    seen: set[str] = set()
    for page in pages:
        if not isinstance(page, dict):
            raise RequestError("page must be an object")
        key, label, aliases = page.get("key"), page.get("label"), page.get("aliases", [])
        if not isinstance(key, str) or not _KEY.fullmatch(key) or key in seen:
            raise RequestError("invalid or duplicate page key")
        if not isinstance(label, str) or len(label) > 80 or "<" in label:
            raise RequestError("invalid page label")
        if not isinstance(aliases, list) or not all(
            isinstance(a, str) and len(a) <= 60 and "<" not in a for a in aliases
        ):
            raise RequestError("invalid aliases")
        seen.add(key)
        clean.append({"key": key, "label": label, "aliases": aliases[:8]})
    actions = payload.get("actions", list(ACTION_ORDER))
    if not isinstance(actions, list) or not all(isinstance(a, str) and a in ACTION_SPECS for a in actions):
        raise RequestError("unknown action")
    return " ".join(text.split()), clean, [a for a in ACTION_ORDER if a in actions]


In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/copilot_model.py
from __future__ import annotations

import math
import threading
import time
from typing import Any, Callable, Optional

from . import copilot_schema as schema

Generate = Callable[[str, "list[dict[str, Any]]", "list[str]"], "tuple[str, float]"]

END_CALL_ID = 49
MAX_NEW_TOKENS = 64


def interpret(generate: Generate, payload: Any, threshold: float) -> "dict[str, Any]":
    started = time.perf_counter()
    text, pages, actions = schema.validate_request(payload)
    output, confidence = generate(text, pages, actions)
    call = schema.parse_call(schema.PREFILL + output.removeprefix(schema.PREFILL), pages, actions)
    if call is None or call.name == schema.NO_MATCH or confidence < threshold:
        call = None
    return {
        "call": None if call is None else call.as_dict(),
        "confidence": confidence if call else 0.0,
        "latencyMs": int((time.perf_counter() - started) * 1000),
    }


class CopilotGenerator:
    def __init__(
        self,
        model_dir: str,
        adapter_dir: Optional[str] = None,
        max_seconds: float = 4.0,
        device: Optional[str] = None,
    ) -> None:
        self._dir = model_dir
        self.adapter_dir = adapter_dir
        self._max_seconds = max_seconds
        self._device = device
        self._model: Any = None
        self._tok: Any = None
        self._torch: Any = None
        self._lock = threading.Lock()

    def _load(self) -> None:
        import torch
        from transformers import AutoModelForCausalLM, AutoTokenizer

        self._torch = torch
        self._tok = AutoTokenizer.from_pretrained(self._dir, local_files_only=True)
        if self._tok.convert_tokens_to_ids(schema.END_CALL) != END_CALL_ID:
            raise RuntimeError("unexpected tokenizer: <end_function_call> id changed")
        model = AutoModelForCausalLM.from_pretrained(self._dir, local_files_only=True, dtype=torch.float32)
        if self.adapter_dir:
            from peft import PeftModel

            model = PeftModel.from_pretrained(model, self.adapter_dir, local_files_only=True).merge_and_unload()
        model = model.eval()
        device = self._device or ("cuda" if torch.cuda.is_available() else "cpu")
        self._model = model.to(device)

    @property
    def device(self) -> str:
        if self._model is None:
            with self._lock:
                if self._model is None:
                    self._load()
        return str(self._model.device)

    def __call__(self, text: str, pages: "list[dict[str, Any]]", actions: "list[str]") -> "tuple[str, float]":
        with self._lock:
            if self._model is None:
                self._load()
            torch, tok, model = self._torch, self._tok, self._model
            prompt = schema.render_prompt(tok, text, pages, actions)
            inputs = tok(prompt, return_tensors="pt", add_special_tokens=False)
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            with torch.inference_mode():
                out = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW_TOKENS,
                    max_time=self._max_seconds,
                    do_sample=False,
                    output_scores=True,
                    return_dict_in_generate=True,
                    eos_token_id=[END_CALL_ID, tok.eos_token_id],
                    pad_token_id=tok.pad_token_id,
                )
            new = out.sequences[0][inputs["input_ids"].shape[-1]:]
            logp = 0.0
            leading = True
            for step, token in enumerate(new):
                if leading and not tok.decode([int(token)]).strip():
                    continue
                leading = False
                probs = torch.softmax(out.scores[step][0].float(), dim=-1)
                logp += math.log(max(float(probs[token]), 1e-12))
            decoded = tok.decode(new, skip_special_tokens=False)
            return decoded, math.exp(logp)


In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/evaluate_copilot.py
from __future__ import annotations

import argparse
import json
import statistics
import time
from pathlib import Path
from typing import Any, Optional

from tools.voice.dashboard_llm import copilot_schema as schema
from tools.voice.dashboard_llm.train import make_dataset as md

HERE = Path(__file__).resolve().parent
DATA = HERE / "train" / "data_copilot"
THRESHOLDS = (0.0, 0.5, 0.7, 0.9, 0.95, 0.98, 0.99, 0.995, 0.999)
REPORT_AT = 0.98


def same_call(a: Optional[dict], b: Optional[dict]) -> bool:
    if a is None or b is None:
        return a is b
    if a["name"] != b["name"] or set(a["args"]) != set(b["args"]):
        return False
    for key, value in a["args"].items():
        other = b["args"][key]
        if isinstance(value, str) and isinstance(other, str):
            if value.casefold() != other.casefold():
                return False
        elif value != other:
            return False
    return True


def _pred(row: dict, threshold: float) -> Optional[dict]:
    return row["call"] if row["confidence"] >= threshold else None


def score_rows(rows: list[dict], threshold: float) -> dict[str, Any]:
    pos = [r for r in rows if r["expect"] is not None]
    neg = [r for r in rows if r["expect"] is None]
    preds = [(r, _pred(r, threshold)) for r in rows]
    wrong = sum(1 for r, p in preds if p is not None and not same_call(p, r["expect"]))
    answered_right = sum(1 for r in pos if same_call(_pred(r, threshold), r["expect"]))
    name_right = sum(
        1 for r in pos if (p := _pred(r, threshold)) is not None and p["name"] == r["expect"]["name"]
    )
    return {
        "threshold": threshold,
        "cases": len(rows),
        "wrongCall": round(wrong / max(1, len(rows)), 4),
        "wrongCount": wrong,
        "yield": round(answered_right / max(1, len(pos)), 4),
        "nameYield": round(name_right / max(1, len(pos)), 4),
        "abstainOnNegatives": round(sum(1 for r in neg if _pred(r, threshold) is None) / max(1, len(neg)), 4),
        "answerable": len(pos),
        "negatives": len(neg),
    }


def breakdown(rows: list[dict], threshold: float, key: str) -> dict[str, dict[str, Any]]:
    groups: dict[str, list[dict]] = {}
    for row in rows:
        groups.setdefault(str(row.get(key, "")), []).append(row)
    return {name: score_rows(group, threshold) for name, group in sorted(groups.items())}


def _pages(keys: list[str]) -> list[dict]:
    return md.request_pages({"pages": keys, "aliasSeed": 0})


def catalogue(name: str) -> list[dict]:
    admin = [k for k in md.ORDER if not k.startswith("team_")]
    keys = {"admin": admin, "supervisor": md.ROLES["supervisor"], "manager": md.ROLES["manager"]}[name]
    return _pages(keys)


def load_frozen(path: Path = HERE / "copilot_eval_phrases.json") -> list[dict]:
    cases = json.loads(path.read_text(encoding="utf-8"))["cases"]
    return [
        {
            "text": c["text"], "lang": c["lang"], "expect": c["expect"],
            "kind": (c["expect"] or {"name": "no_match"})["name"],
            "request": {"text": c["text"], "pages": catalogue(c["catalogue"]),
                        "actions": c.get("actions", list(schema.ACTION_ORDER))},
        }
        for c in cases
    ]


def load_test(data_dir: Path = DATA, split: str = "test") -> list[dict]:
    from tools.voice.dashboard_llm.train import make_copilot_dataset as ds

    rows = [json.loads(x) for x in (data_dir / f"{split}.jsonl").read_text(encoding="utf-8").splitlines()]
    return [
        {
            "text": r["text"], "lang": r["lang"], "kind": r["kind"].split(":")[0],
            "expect": None if r["target"]["name"] == schema.NO_MATCH else r["target"],
            "request": ds.to_request(r),
        }
        for r in rows
    ]


def load_nav80(path: Path = HERE / "eval_phrases.json") -> list[dict]:
    data = json.loads(path.read_text(encoding="utf-8"))
    out = []
    for c in data["cases"]:
        if c.get("kind") == "compound":
            continue
        expect = c.get("expect")
        out.append({
            "text": c["text"], "lang": "en", "kind": "nav",
            "expect": None if expect in (None, md.NO_MATCH) else {"name": schema.NAVIGATE, "args": {"page": expect}},
            "request": {"text": c["text"], "pages": data["catalogues"][c["catalogue"]],
                        "actions": list(schema.ACTION_ORDER)},
        })
    return out


def run(generate, cases: list[dict]) -> list[dict]:
    from tools.voice.dashboard_llm import copilot_model

    rows = []
    for case in cases:
        result = copilot_model.interpret(generate, case["request"], 0.0)
        rows.append({k: v for k, v in case.items() if k != "request"} | result)
    return rows


def summarise(rows: list[dict]) -> dict[str, Any]:
    latency = sorted(r["latencyMs"] for r in rows)
    return {
        "latencyMs": {"p50": int(statistics.median(latency)), "p95": latency[max(0, int(len(latency) * 0.95) - 1)]},
        "sweep": [score_rows(rows, t) for t in THRESHOLDS],
        f"byLang@{REPORT_AT}": breakdown(rows, REPORT_AT, "lang"),
        f"byKind@{REPORT_AT}": breakdown(rows, REPORT_AT, "kind"),
        "rows": rows,
    }


def main(argv: Optional[list[str]] = None) -> None:
    from tools.voice.dashboard_llm import copilot_model

    ap = argparse.ArgumentParser()
    ap.add_argument("--model-dir", required=True)
    ap.add_argument("--adapter-dir")
    ap.add_argument("--no-base", action="store_true", help="skip the untuned base model")
    ap.add_argument("--data-dir", type=Path, default=DATA)
    ap.add_argument("--sets", default="frozen,test,nav80")
    ap.add_argument("--limit", type=int, default=0, help="first N cases per set (smoke test)")
    ap.add_argument("--device")
    ap.add_argument("--output", type=Path, required=True)
    args = ap.parse_args(argv)

    loaders = {
        "frozen": load_frozen,
        "test": lambda: load_test(args.data_dir),
        "validation": lambda: load_test(args.data_dir, "validation"),
        "nav80": load_nav80,
    }
    sets = {name: loaders[name]() for name in args.sets.split(",")}
    if args.limit:
        sets = {name: cases[: args.limit] for name, cases in sets.items()}
    variants: dict[str, Optional[str]] = {} if args.no_base else {"base": None}
    if args.adapter_dir:
        variants["adapter"] = args.adapter_dir

    report: dict[str, Any] = {"ranAt": time.strftime("%Y-%m-%dT%H:%M:%S"), "schema": "copilot-eval-report/1", "results": {}}
    for name, adapter in variants.items():
        generate = copilot_model.CopilotGenerator(args.model_dir, adapter_dir=adapter, device=args.device)
        generate("warm up", [], list(schema.ACTION_ORDER))
        report["device"] = generate.device
        for set_name, cases in sets.items():
            rows = run(generate, cases)
            report["results"][f"{name}/{set_name}"] = summarise(rows)
            print(name, set_name, report["results"][f"{name}/{set_name}"]["latencyMs"], flush=True)
        del generate
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(report, indent=1, ensure_ascii=False), encoding="utf-8")
    for key, result in report["results"].items():
        print(key)
        for line in result["sweep"]:
            print(" ", line)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/eval_phrases.json
{
 "catalogues": {
  "admin": [
   {
    "key": "dashboard",
    "label": "Dashboard",
    "aliases": [
     "home",
     "overview",
     "kpi",
     "summary"
    ]
   },
   {
    "key": "salesman_activity",
    "label": "Salesman Tracking",
    "aliases": [
     "sales activity",
     "map",
     "gps",
     "location",
     "live",
     "performance"
    ]
   },
   {
    "key": "visit_logs",
    "label": "Visit Logs",
    "aliases": [
     "visits",
     "visit history",
     "arrival",
     "follow-up"
    ]
   },
   {
    "key": "route_planner",
    "label": "Route Planner",
    "aliases": [
     "routes",
     "weekly routes",
     "route planning",
     "plan"
    ]
   },
   {
    "key": "notifications",
    "label": "Notifications",
    "aliases": [
     "alerts",
     "feed"
    ]
   },
   {
    "key": "all_orders",
    "label": "All Orders",
    "aliases": [
     "orders",
     "submitted orders",
     "order approval",
     "sales order",
     "approve",
     "reject"
    ]
   },
   {
    "key": "purchase_order",
    "label": "Purchase Orders",
    "aliases": [
     "procurement",
     "po",
     "supply"
    ]
   },
   {
    "key": "special_requests",
    "label": "Special Requests",
    "aliases": [
     "exceptions",
     "request approvals"
    ]
   },
   {
    "key": "reports",
    "label": "Reports",
    "aliases": [
     "analytics",
     "monthly analytics"
    ]
   },
   {
    "key": "prediction",
    "label": "Prediction",
    "aliases": [
     "forecast",
     "demand forecast",
     "predictions"
    ]
   },
   {
    "key": "trust_impact",
    "label": "Trust & Impact",
    "aliases": [
     "evidence",
     "abstentions",
     "accuracy"
    ]
   },
   {
    "key": "product",
    "label": "Products",
    "aliases": [
     "product catalogue",
     "catalogue"
    ]
   },
   {
    "key": "retailer",
    "label": "Retailers",
    "aliases": [
     "outlets"
    ]
   },
   {
    "key": "user_directory",
    "label": "Users",
    "aliases": [
     "staff directory"
    ]
   },
   {
    "key": "users",
    "label": "Role Assignment",
    "aliases": [
     "user management",
     "roles",
     "accounts"
    ]
   },
   {
    "key": "branches",
    "label": "Branches",
    "aliases": [
     "branch management"
    ]
   },
   {
    "key": "coordinate_review",
    "label": "Coordinate Review",
    "aliases": [
     "gps",
     "latitude",
     "geocode"
    ]
   },
   {
    "key": "import_preview",
    "label": "Import Centre",
    "aliases": [
     "csv",
     "excel"
    ]
   },
   {
    "key": "backup",
    "label": "Backup",
    "aliases": []
   },
   {
    "key": "system_health",
    "label": "System Health",
    "aliases": []
   }
  ],
  "branch_manager": [
   {
    "key": "dashboard",
    "label": "Dashboard",
    "aliases": [
     "home",
     "overview",
     "kpi",
     "summary"
    ]
   },
   {
    "key": "salesman_activity",
    "label": "Salesman Tracking",
    "aliases": [
     "sales activity",
     "map",
     "gps",
     "location",
     "live",
     "performance"
    ]
   },
   {
    "key": "visit_logs",
    "label": "Visit Logs",
    "aliases": [
     "visits",
     "visit history",
     "arrival",
     "follow-up"
    ]
   },
   {
    "key": "route_planner",
    "label": "Route Planner",
    "aliases": [
     "routes",
     "weekly routes",
     "route planning",
     "plan"
    ]
   },
   {
    "key": "notifications",
    "label": "Notifications",
    "aliases": [
     "alerts",
     "feed"
    ]
   },
   {
    "key": "all_orders",
    "label": "All Orders",
    "aliases": [
     "orders",
     "submitted orders",
     "order approval",
     "sales order",
     "approve",
     "reject"
    ]
   },
   {
    "key": "purchase_order",
    "label": "Purchase Orders",
    "aliases": [
     "procurement",
     "po",
     "supply"
    ]
   },
   {
    "key": "special_requests",
    "label": "Special Requests",
    "aliases": [
     "exceptions",
     "request approvals"
    ]
   },
   {
    "key": "reports",
    "label": "Reports",
    "aliases": [
     "analytics",
     "monthly analytics"
    ]
   },
   {
    "key": "prediction",
    "label": "Prediction",
    "aliases": [
     "forecast",
     "demand forecast",
     "predictions"
    ]
   }
  ]
 },
 "cases": [
  {
   "text": "i wanna check how my reps are doing today",
   "catalogue": "admin",
   "expect": "salesman_activity",
   "kind": "natural"
  },
  {
   "text": "where are my salesmen right now",
   "catalogue": "admin",
   "expect": "salesman_activity",
   "kind": "natural"
  },
  {
   "text": "show me where the sales team is",
   "catalogue": "admin",
   "expect": "salesman_activity",
   "kind": "natural"
  },
  {
   "text": "who has visited which shop",
   "catalogue": "admin",
   "expect": "visit_logs",
   "kind": "natural"
  },
  {
   "text": "i need to see the visit history",
   "catalogue": "admin",
   "expect": "visit_logs",
   "kind": "natural"
  },
  {
   "text": "what orders are waiting for me",
   "catalogue": "admin",
   "expect": "all_orders",
   "kind": "natural"
  },
  {
   "text": "i have some orders to approve",
   "catalogue": "admin",
   "expect": "all_orders",
   "kind": "natural"
  },
  {
   "text": "can you open the orders for me",
   "catalogue": "admin",
   "expect": "all_orders",
   "kind": "natural"
  },
  {
   "text": "let me look at the sales orders",
   "catalogue": "admin",
   "expect": "all_orders",
   "kind": "natural"
  },
  {
   "text": "i need to plan the routes for next week",
   "catalogue": "admin",
   "expect": "route_planner",
   "kind": "natural"
  },
  {
   "text": "open the weekly route thing",
   "catalogue": "admin",
   "expect": "route_planner",
   "kind": "natural"
  },
  {
   "text": "take me to the map of my drivers",
   "catalogue": "admin",
   "expect": "salesman_activity",
   "kind": "natural"
  },
  {
   "text": "what is the demand forecast",
   "catalogue": "admin",
   "expect": "prediction",
   "kind": "natural"
  },
  {
   "text": "how many units will we sell next week",
   "catalogue": "admin",
   "expect": "prediction",
   "kind": "natural"
  },
  {
   "text": "show me the predictions",
   "catalogue": "admin",
   "expect": "prediction",
   "kind": "natural"
  },
  {
   "text": "i want to see the monthly analytics",
   "catalogue": "admin",
   "expect": "reports",
   "kind": "natural"
  },
  {
   "text": "pull up the reports",
   "catalogue": "admin",
   "expect": "reports",
   "kind": "natural"
  },
  {
   "text": "any new alerts",
   "catalogue": "admin",
   "expect": "notifications",
   "kind": "natural"
  },
  {
   "text": "check my notifications",
   "catalogue": "admin",
   "expect": "notifications",
   "kind": "natural"
  },
  {
   "text": "go back home",
   "catalogue": "admin",
   "expect": "dashboard",
   "kind": "natural"
  },
  {
   "text": "take me to the main overview",
   "catalogue": "admin",
   "expect": "dashboard",
   "kind": "natural"
  },
  {
   "text": "open the home page",
   "catalogue": "admin",
   "expect": "dashboard",
   "kind": "natural"
  },
  {
   "text": "i need to create a purchase order",
   "catalogue": "admin",
   "expect": "purchase_order",
   "kind": "natural"
  },
  {
   "text": "show procurement",
   "catalogue": "admin",
   "expect": "purchase_order",
   "kind": "natural"
  },
  {
   "text": "are there any special requests to approve",
   "catalogue": "admin",
   "expect": "special_requests",
   "kind": "natural"
  },
  {
   "text": "open exceptions",
   "catalogue": "admin",
   "expect": "special_requests",
   "kind": "natural"
  },
  {
   "text": "show me the product catalogue",
   "catalogue": "admin",
   "expect": "product",
   "kind": "natural"
  },
  {
   "text": "i want to look at the products",
   "catalogue": "admin",
   "expect": "product",
   "kind": "natural"
  },
  {
   "text": "list all the outlets",
   "catalogue": "admin",
   "expect": "retailer",
   "kind": "natural"
  },
  {
   "text": "open retailers",
   "catalogue": "admin",
   "expect": "retailer",
   "kind": "natural"
  },
  {
   "text": "who are the staff",
   "catalogue": "admin",
   "expect": "user_directory",
   "kind": "natural"
  },
  {
   "text": "show the staff directory",
   "catalogue": "admin",
   "expect": "user_directory",
   "kind": "natural"
  },
  {
   "text": "change someones role",
   "catalogue": "admin",
   "expect": "users",
   "kind": "natural"
  },
  {
   "text": "i need to assign roles to people",
   "catalogue": "admin",
   "expect": "users",
   "kind": "natural"
  },
  {
   "text": "show me the branches",
   "catalogue": "admin",
   "expect": "branches",
   "kind": "natural"
  },
  {
   "text": "review the gps coordinates of outlets",
   "catalogue": "admin",
   "expect": "coordinate_review",
   "kind": "natural"
  },
  {
   "text": "i want to import a csv",
   "catalogue": "admin",
   "expect": "import_preview",
   "kind": "natural"
  },
  {
   "text": "upload an excel file",
   "catalogue": "admin",
   "expect": "import_preview",
   "kind": "natural"
  },
  {
   "text": "is the system healthy",
   "catalogue": "admin",
   "expect": "system_health",
   "kind": "natural"
  },
  {
   "text": "open backup",
   "catalogue": "admin",
   "expect": "backup",
   "kind": "natural"
  },
  {
   "text": "show trust and impact",
   "catalogue": "admin",
   "expect": "trust_impact",
   "kind": "natural"
  },
  {
   "text": "how accurate are the forecasts",
   "catalogue": "admin",
   "expect": "trust_impact",
   "kind": "natural"
  },
  {
   "text": "open the ordrs",
   "catalogue": "admin",
   "expect": "all_orders",
   "kind": "typo"
  },
  {
   "text": "show salesmen trackin",
   "catalogue": "admin",
   "expect": "salesman_activity",
   "kind": "typo"
  },
  {
   "text": "go to reprots",
   "catalogue": "admin",
   "expect": "reports",
   "kind": "typo"
  },
  {
   "text": "opne notifcations",
   "catalogue": "admin",
   "expect": "notifications",
   "kind": "typo"
  },
  {
   "text": "route plaanner please",
   "catalogue": "admin",
   "expect": "route_planner",
   "kind": "typo"
  },
  {
   "text": "show me visit lgos",
   "catalogue": "admin",
   "expect": "visit_logs",
   "kind": "typo"
  },
  {
   "text": "purchse orders",
   "catalogue": "admin",
   "expect": "purchase_order",
   "kind": "typo"
  },
  {
   "text": "sales man activity",
   "catalogue": "admin",
   "expect": "salesman_activity",
   "kind": "typo"
  },
  {
   "text": "open system health",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "role_gated"
  },
  {
   "text": "show me the backup",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "role_gated"
  },
  {
   "text": "i want to import a csv",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "role_gated"
  },
  {
   "text": "open role assignment",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "role_gated"
  },
  {
   "text": "show the staff directory",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "role_gated"
  },
  {
   "text": "open trust and impact",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "role_gated"
  },
  {
   "text": "review the gps coordinates of outlets",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "role_gated"
  },
  {
   "text": "show me the products",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "role_gated"
  },
  {
   "text": "show me the orders",
   "catalogue": "branch_manager",
   "expect": "all_orders",
   "kind": "role_ok"
  },
  {
   "text": "where are my reps",
   "catalogue": "branch_manager",
   "expect": "salesman_activity",
   "kind": "role_ok"
  },
  {
   "text": "plan the routes",
   "catalogue": "branch_manager",
   "expect": "route_planner",
   "kind": "role_ok"
  },
  {
   "text": "what is the weather today",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "tell me a joke",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "delete all the orders",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "reset everyone's password",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "send an email to the supplier",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "what time is it",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "how do i cook rice",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "approve order 1234",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "who won the football match",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "sing me a song",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "hello",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "um",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "banana",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "turn off the lights",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "call my manager",
   "catalogue": "admin",
   "expect": null,
   "kind": "negative"
  },
  {
   "text": "open orders and then open reports",
   "catalogue": "admin",
   "expect": null,
   "kind": "compound"
  },
  {
   "text": "show the map and then go to notifications",
   "catalogue": "admin",
   "expect": null,
   "kind": "compound"
  },
  {
   "text": "open products and check the retailers",
   "catalogue": "admin",
   "expect": null,
   "kind": "compound"
  },
  {
   "text": "go to visit logs then open the route planner",
   "catalogue": "admin",
   "expect": null,
   "kind": "compound"
  }
 ]
}


In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/fresh_phrases.json
{
 "catalogues": {
  "admin": [
   {
    "key": "dashboard",
    "label": "Dashboard",
    "aliases": [
     "home",
     "overview",
     "kpi",
     "summary"
    ]
   },
   {
    "key": "salesman_activity",
    "label": "Salesman Tracking",
    "aliases": [
     "sales activity",
     "map",
     "gps",
     "location",
     "live",
     "performance"
    ]
   },
   {
    "key": "visit_logs",
    "label": "Visit Logs",
    "aliases": [
     "visits",
     "visit history",
     "arrival",
     "follow-up"
    ]
   },
   {
    "key": "route_planner",
    "label": "Route Planner",
    "aliases": [
     "routes",
     "weekly routes",
     "route planning",
     "plan"
    ]
   },
   {
    "key": "notifications",
    "label": "Notifications",
    "aliases": [
     "alerts",
     "feed"
    ]
   },
   {
    "key": "all_orders",
    "label": "All Orders",
    "aliases": [
     "orders",
     "submitted orders",
     "order approval",
     "sales order",
     "approve",
     "reject"
    ]
   },
   {
    "key": "purchase_order",
    "label": "Purchase Orders",
    "aliases": [
     "procurement",
     "po",
     "supply"
    ]
   },
   {
    "key": "special_requests",
    "label": "Special Requests",
    "aliases": [
     "exceptions",
     "request approvals"
    ]
   },
   {
    "key": "reports",
    "label": "Reports",
    "aliases": [
     "analytics",
     "monthly analytics"
    ]
   },
   {
    "key": "prediction",
    "label": "Prediction",
    "aliases": [
     "forecast",
     "demand forecast",
     "predictions"
    ]
   },
   {
    "key": "trust_impact",
    "label": "Trust & Impact",
    "aliases": [
     "evidence",
     "abstentions",
     "accuracy"
    ]
   },
   {
    "key": "product",
    "label": "Products",
    "aliases": [
     "product catalogue",
     "catalogue"
    ]
   },
   {
    "key": "department",
    "label": "Departments",
    "aliases": []
   },
   {
    "key": "retailer",
    "label": "Retailers",
    "aliases": [
     "outlets"
    ]
   },
   {
    "key": "user_directory",
    "label": "Users",
    "aliases": [
     "staff directory"
    ]
   },
   {
    "key": "users",
    "label": "Role Assignment",
    "aliases": [
     "user management",
     "roles",
     "accounts"
    ]
   },
   {
    "key": "permissions",
    "label": "Role Permissions",
    "aliases": [
     "permission",
     "matrix",
     "access",
     "authorization"
    ]
   },
   {
    "key": "branches",
    "label": "Branches",
    "aliases": [
     "branch management"
    ]
   },
   {
    "key": "coordinate_review",
    "label": "Coordinate Review",
    "aliases": [
     "gps",
     "latitude",
     "geocode"
    ]
   },
   {
    "key": "import_preview",
    "label": "Import Centre",
    "aliases": [
     "csv",
     "excel"
    ]
   },
   {
    "key": "backup",
    "label": "Backup",
    "aliases": []
   },
   {
    "key": "privacy_policy",
    "label": "Privacy Policy",
    "aliases": []
   },
   {
    "key": "audit_log",
    "label": "Audit Log",
    "aliases": [
     "history",
     "changes",
     "actor",
     "governance"
    ]
   },
   {
    "key": "system_health",
    "label": "System Health",
    "aliases": []
   }
  ],
  "branch_manager": [
   {
    "key": "dashboard",
    "label": "Dashboard",
    "aliases": [
     "home",
     "overview",
     "kpi",
     "summary"
    ]
   },
   {
    "key": "salesman_activity",
    "label": "Salesman Tracking",
    "aliases": [
     "sales activity",
     "map",
     "gps",
     "location",
     "live",
     "performance"
    ]
   },
   {
    "key": "visit_logs",
    "label": "Visit Logs",
    "aliases": [
     "visits",
     "visit history",
     "arrival",
     "follow-up"
    ]
   },
   {
    "key": "route_planner",
    "label": "Route Planner",
    "aliases": [
     "routes",
     "weekly routes",
     "route planning",
     "plan"
    ]
   },
   {
    "key": "notifications",
    "label": "Notifications",
    "aliases": [
     "alerts",
     "feed"
    ]
   },
   {
    "key": "all_orders",
    "label": "All Orders",
    "aliases": [
     "orders",
     "submitted orders",
     "order approval",
     "sales order",
     "approve",
     "reject"
    ]
   },
   {
    "key": "purchase_order",
    "label": "Purchase Orders",
    "aliases": [
     "procurement",
     "po",
     "supply"
    ]
   },
   {
    "key": "special_requests",
    "label": "Special Requests",
    "aliases": [
     "exceptions",
     "request approvals"
    ]
   },
   {
    "key": "reports",
    "label": "Reports",
    "aliases": [
     "analytics",
     "monthly analytics"
    ]
   },
   {
    "key": "prediction",
    "label": "Prediction",
    "aliases": [
     "forecast",
     "demand forecast",
     "predictions"
    ]
   }
  ]
 },
 "cases": [
  {
   "text": "how is the field team getting on this morning",
   "catalogue": "admin",
   "expect": "salesman_activity",
   "kind": "fresh"
  },
  {
   "text": "is anybody from the sales team at a shop right now",
   "catalogue": "admin",
   "expect": "salesman_activity",
   "kind": "fresh"
  },
  {
   "text": "which stores were covered by the guys yesterday",
   "catalogue": "admin",
   "expect": "visit_logs",
   "kind": "fresh"
  },
  {
   "text": "anything from suppliers i should be ordering",
   "catalogue": "admin",
   "expect": "purchase_order",
   "kind": "fresh"
  },
  {
   "text": "the shops sent in some new orders",
   "catalogue": "admin",
   "expect": "all_orders",
   "kind": "fresh"
  },
  {
   "text": "what is coming next month in terms of demand",
   "catalogue": "admin",
   "expect": "prediction",
   "kind": "fresh"
  },
  {
   "text": "i need to see who can open which screens",
   "catalogue": "admin",
   "expect": "permissions",
   "kind": "fresh"
  },
  {
   "text": "has the backup run lately",
   "catalogue": "admin",
   "expect": "backup",
   "kind": "fresh"
  },
  {
   "text": "show me the numbers for last month",
   "catalogue": "admin",
   "expect": "reports",
   "kind": "fresh"
  },
  {
   "text": "lay out next week's trips for the team",
   "catalogue": "admin",
   "expect": "route_planner",
   "kind": "fresh"
  },
  {
   "text": "some shop wants a special deal, where do i look",
   "catalogue": "admin",
   "expect": "special_requests",
   "kind": "fresh"
  },
  {
   "text": "who touched the settings last night",
   "catalogue": "admin",
   "expect": "audit_log",
   "kind": "fresh"
  },
  {
   "text": "i need to bulk load a spreadsheet of retailers",
   "catalogue": "admin",
   "expect": "import_preview",
   "kind": "fresh"
  },
  {
   "text": "where do i read the privacy stuff",
   "catalogue": "admin",
   "expect": "privacy_policy",
   "kind": "fresh"
  },
  {
   "text": "what happened while i was away",
   "catalogue": "admin",
   "expect": "notifications",
   "kind": "fresh"
  },
  {
   "text": "the retailers pins on the map look wrong",
   "catalogue": "admin",
   "expect": "coordinate_review",
   "kind": "fresh"
  },
  {
   "text": "i need to give a new guy admin access",
   "catalogue": "admin",
   "expect": "users",
   "kind": "fresh"
  },
  {
   "text": "what departments do we have",
   "catalogue": "admin",
   "expect": "department",
   "kind": "fresh"
  },
  {
   "text": "what items do we stock",
   "catalogue": "admin",
   "expect": "product",
   "kind": "fresh"
  },
  {
   "text": "what is the weather like in kuching",
   "catalogue": "admin",
   "expect": null,
   "kind": "fresh"
  },
  {
   "text": "wipe the entire order history",
   "catalogue": "admin",
   "expect": null,
   "kind": "fresh"
  },
  {
   "text": "thanks that was helpful",
   "catalogue": "admin",
   "expect": null,
   "kind": "fresh"
  },
  {
   "text": "open the payroll screen",
   "catalogue": "admin",
   "expect": null,
   "kind": "fresh"
  },
  {
   "text": "can i open the database backup",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "fresh"
  },
  {
   "text": "i need to manage the user accounts",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "fresh"
  },
  {
   "text": "where is the staff list",
   "catalogue": "branch_manager",
   "expect": null,
   "kind": "fresh"
  },
  {
   "text": "go to reports and then to the visit logs",
   "catalogue": "admin",
   "expect": null,
   "kind": "fresh"
  },
  {
   "text": "mark all special requests as approved",
   "catalogue": "admin",
   "expect": null,
   "kind": "fresh"
  },
  {
   "text": "hello there how is it going",
   "catalogue": "admin",
   "expect": null,
   "kind": "fresh"
  },
  {
   "text": "show me the stock in the warehouse",
   "catalogue": "admin",
   "expect": null,
   "kind": "fresh"
  }
 ]
}


In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/copilot_eval_phrases.json
{
  "description": "Frozen copilot evaluation set. Written by hand BEFORE training the copilot adapter. Never used for training, validation or tuning; make_copilot_dataset.py refuses any training sentence that equals one of these texts. Names are not in any training pool. expect=null means the model must decline (no_match). 'actions' omitted = all six actions offered. 'catalogue' names a page list built in evaluate_copilot.py.",
  "version": "copilot-eval/1.0.0",
  "cases": [
    {"text": "show Hafizah's orders this week", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Hafizah", "period": "this_week"}}},
    {"text": "show me Boon Keat's approved orders for last month", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Boon Keat", "status": "approved", "period": "last_month"}}},
    {"text": "orders from Kedai Bestari yesterday", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"retailer": "Kedai Bestari", "period": "yesterday"}}},
    {"text": "list rejected orders for Pasar Mini Cempaka", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"status": "rejected", "retailer": "Pasar Mini Cempaka"}}},
    {"text": "how many orders did Gopal submit today", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Gopal", "period": "today"}}},
    {"text": "find submitted orders from Hock Lee Provision this month", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"status": "submitted", "retailer": "Hock Lee Provision", "period": "this_month"}}},
    {"text": "what did Shazwan sell last week", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Shazwan", "period": "last_week"}}},
    {"text": "pull up pending orders from Kedai Runcit Salmah", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"status": "submitted", "retailer": "Kedai Runcit Salmah"}}},
    {"text": "any orders from Taman Desa Mart this week", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"retailer": "Taman Desa Mart", "period": "this_week"}}},
    {"text": "check Julia's declined orders", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Julia", "status": "rejected"}}},
    {"text": "show orders today", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"period": "today"}}},
    {"text": "orders last month", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"period": "last_month"}}},
    {"text": "show me new orders from Orchid Minimart this week", "catalogue": "supervisor", "lang": "en", "expect": {"name": "search_orders", "args": {"status": "submitted", "retailer": "Orchid Minimart", "period": "this_week"}}},
    {"text": "what has Wan Aiman sold so far today", "catalogue": "supervisor", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Wan Aiman", "period": "today"}}},
    {"text": "did Bukit Mas Grocer order anything last week", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"retailer": "Bukit Mas Grocer", "period": "last_week"}}},
    {"text": "i want to see Chin's accepted orders at Fresh Choice Mart", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Chin", "status": "approved", "retailer": "Fresh Choice Mart"}}},
    {"text": "search orders by Rosnah", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Rosnah"}}},
    {"text": "get me yesterday's orders from Kedai Runcit Pak Din", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"retailer": "Kedai Runcit Pak Din", "period": "yesterday"}}},
    {"text": "how many approved orders this month", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"status": "approved", "period": "this_month"}}},
    {"text": "show Yusuf's orders for Kedai Bestari", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Yusuf", "retailer": "Kedai Bestari"}}},
    {"text": "list the orders Sook Fun submitted over this week", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Sook Fun", "period": "this_week"}}},
    {"text": "tunjuk pesanan Hafizah minggu ini", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"salesperson": "Hafizah", "period": "this_week"}}},
    {"text": "pesanan diluluskan dari Kedai Bestari bulan lepas", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"status": "approved", "retailer": "Kedai Bestari", "period": "last_month"}}},
    {"text": "senaraikan pesanan Gopal hari ini", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"salesperson": "Gopal", "period": "today"}}},
    {"text": "cari pesanan ditolak untuk Taman Desa Mart", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"status": "rejected", "retailer": "Taman Desa Mart"}}},
    {"text": "berapa pesanan Shazwan semalam", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"salesperson": "Shazwan", "period": "yesterday"}}},
    {"text": "papar pesanan dihantar minggu lalu", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"status": "submitted", "period": "last_week"}}},
    {"text": "saya nak tengok pesanan Julia bulan ini", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"salesperson": "Julia", "period": "this_month"}}},
    {"text": "ada pesanan dari Orchid Minimart hari ini", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"retailer": "Orchid Minimart", "period": "today"}}},
    {"text": "pesanan baharu minggu ini", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"status": "submitted", "period": "this_week"}}},
    {"text": "tunjuk pesanan Rosnah di Bukit Mas Grocer", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"salesperson": "Rosnah", "retailer": "Bukit Mas Grocer"}}},
    {"text": "lihat pesanan ditolak sepanjang minggu ini", "catalogue": "supervisor", "lang": "ms", "expect": {"name": "search_orders", "args": {"status": "rejected", "period": "this_week"}}},
    {"text": "show pesanan Boon Keat today", "catalogue": "admin", "lang": "mix", "expect": {"name": "search_orders", "args": {"salesperson": "Boon Keat", "period": "today"}}},
    {"text": "tunjuk orders Kedai Bestari this week", "catalogue": "admin", "lang": "mix", "expect": {"name": "search_orders", "args": {"retailer": "Kedai Bestari", "period": "this_week"}}},
    {"text": "orders Hafizah bulan ini", "catalogue": "admin", "lang": "mix", "expect": {"name": "search_orders", "args": {"salesperson": "Hafizah", "period": "this_month"}}},
    {"text": "check pesanan Gopal semalam", "catalogue": "admin", "lang": "mix", "expect": {"name": "search_orders", "args": {"salesperson": "Gopal", "period": "yesterday"}}},
    {"text": "list approved pesanan minggu lepas", "catalogue": "admin", "lang": "mix", "expect": {"name": "search_orders", "args": {"status": "approved", "period": "last_week"}}},
    {"text": "orders dari Julia hari ini", "catalogue": "admin", "lang": "mix", "expect": {"name": "search_orders", "args": {"salesperson": "Julia", "period": "today"}}},
    {"text": "only Kedai Bestari", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"retailer": "Kedai Bestari"}}},
    {"text": "just the approved ones", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"status": "approved"}}},
    {"text": "last week instead", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"period": "last_week"}}},
    {"text": "what about Gopal", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"salesperson": "Gopal"}}},
    {"text": "filter by rejected", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"status": "rejected"}}},
    {"text": "make it this month", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"period": "this_month"}}},
    {"text": "how about Hock Lee Provision", "catalogue": "admin", "lang": "en", "expect": {"name": "search_orders", "args": {"retailer": "Hock Lee Provision"}}},
    {"text": "hanya Taman Desa Mart", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"retailer": "Taman Desa Mart"}}},
    {"text": "yang diluluskan sahaja", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"status": "approved"}}},
    {"text": "untuk Shazwan sahaja", "catalogue": "admin", "lang": "ms", "expect": {"name": "search_orders", "args": {"salesperson": "Shazwan"}}},
    {"text": "what do i need to look at today", "catalogue": "admin", "lang": "en", "expect": {"name": "daily_briefing", "args": {}}},
    {"text": "can i have my briefing for today", "catalogue": "supervisor", "lang": "en", "expect": {"name": "daily_briefing", "args": {}}},
    {"text": "what should i look at first this morning", "catalogue": "admin", "lang": "en", "expect": {"name": "daily_briefing", "args": {}}},
    {"text": "any priorities for me today", "catalogue": "supervisor", "lang": "en", "expect": {"name": "daily_briefing", "args": {}}},
    {"text": "start my morning briefing", "catalogue": "admin", "lang": "en", "expect": {"name": "daily_briefing", "args": {}}},
    {"text": "apa yang saya perlu beri perhatian hari ini", "catalogue": "admin", "lang": "ms", "expect": {"name": "daily_briefing", "args": {}}},
    {"text": "tolong beri ringkasan harian saya", "catalogue": "supervisor", "lang": "ms", "expect": {"name": "daily_briefing", "args": {}}},
    {"text": "apa yang penting hari ini", "catalogue": "admin", "lang": "ms", "expect": {"name": "daily_briefing", "args": {}}},
    {"text": "export all of these", "catalogue": "admin", "lang": "en", "expect": {"name": "export_results", "args": {}}},
    {"text": "download the results as a csv file", "catalogue": "admin", "lang": "en", "expect": {"name": "export_results", "args": {}}},
    {"text": "can you save this list to a spreadsheet", "catalogue": "supervisor", "lang": "en", "expect": {"name": "export_results", "args": {}}},
    {"text": "csv please", "catalogue": "admin", "lang": "en", "expect": {"name": "export_results", "args": {}}},
    {"text": "tolong eksport senarai ini", "catalogue": "admin", "lang": "ms", "expect": {"name": "export_results", "args": {}}},
    {"text": "boleh muat turun keputusan ini sebagai csv", "catalogue": "admin", "lang": "ms", "expect": {"name": "export_results", "args": {}}},
    {"text": "explain the second item", "catalogue": "admin", "lang": "en", "expect": {"name": "explain_item", "args": {"position": 2}}},
    {"text": "why is item 4 on the list", "catalogue": "admin", "lang": "en", "expect": {"name": "explain_item", "args": {"position": 4}}},
    {"text": "tell me more about the first one", "catalogue": "supervisor", "lang": "en", "expect": {"name": "explain_item", "args": {"position": 1}}},
    {"text": "what does number 3 mean", "catalogue": "admin", "lang": "en", "expect": {"name": "explain_item", "args": {"position": 3}}},
    {"text": "why is this one important", "catalogue": "admin", "lang": "en", "expect": {"name": "explain_item", "args": {}}},
    {"text": "can you explain that", "catalogue": "supervisor", "lang": "en", "expect": {"name": "explain_item", "args": {}}},
    {"text": "terangkan item ketiga", "catalogue": "admin", "lang": "ms", "expect": {"name": "explain_item", "args": {"position": 3}}},
    {"text": "kenapa item kelima penting", "catalogue": "admin", "lang": "ms", "expect": {"name": "explain_item", "args": {"position": 5}}},
    {"text": "mark the second item as in review", "catalogue": "admin", "lang": "en", "expect": {"name": "set_item_status", "args": {"position": 2, "status": "in_review"}}},
    {"text": "mark it as in review", "catalogue": "admin", "lang": "en", "expect": {"name": "set_item_status", "args": {"status": "in_review"}}},
    {"text": "set item 3 to done", "catalogue": "supervisor", "lang": "en", "expect": {"name": "set_item_status", "args": {"position": 3, "status": "resolved"}}},
    {"text": "ignore the first item", "catalogue": "admin", "lang": "en", "expect": {"name": "set_item_status", "args": {"position": 1, "status": "ignored"}}},
    {"text": "dismiss this one", "catalogue": "admin", "lang": "en", "expect": {"name": "set_item_status", "args": {"status": "ignored"}}},
    {"text": "put the fourth one under review", "catalogue": "admin", "lang": "en", "expect": {"name": "set_item_status", "args": {"position": 4, "status": "in_review"}}},
    {"text": "tandakan item kedua sebagai dalam semakan", "catalogue": "admin", "lang": "ms", "expect": {"name": "set_item_status", "args": {"position": 2, "status": "in_review"}}},
    {"text": "tolong abaikan yang ini", "catalogue": "supervisor", "lang": "ms", "expect": {"name": "set_item_status", "args": {"status": "ignored"}}},
    {"text": "brief me on Kedai Bestari", "catalogue": "admin", "lang": "en", "expect": {"name": "brief_retailer", "args": {"retailer": "Kedai Bestari"}}},
    {"text": "prepare me for my visit to Pasar Mini Cempaka", "catalogue": "admin", "lang": "en", "expect": {"name": "brief_retailer", "args": {"retailer": "Pasar Mini Cempaka"}}},
    {"text": "what should i discuss with Hock Lee Provision", "catalogue": "admin", "lang": "en", "expect": {"name": "brief_retailer", "args": {"retailer": "Hock Lee Provision"}}},
    {"text": "visit brief for Taman Desa Mart", "catalogue": "supervisor", "lang": "en", "expect": {"name": "brief_retailer", "args": {"retailer": "Taman Desa Mart"}}},
    {"text": "i am seeing Orchid Minimart tomorrow, what do i need to know", "catalogue": "admin", "lang": "en", "expect": {"name": "brief_retailer", "args": {"retailer": "Orchid Minimart"}}},
    {"text": "brief me about this retailer", "catalogue": "admin", "lang": "en", "expect": {"name": "brief_retailer", "args": {}}},
    {"text": "give me the background on Bukit Mas Grocer", "catalogue": "admin", "lang": "en", "expect": {"name": "brief_retailer", "args": {"retailer": "Bukit Mas Grocer"}}},
    {"text": "beri taklimat tentang Kedai Runcit Salmah", "catalogue": "admin", "lang": "ms", "expect": {"name": "brief_retailer", "args": {"retailer": "Kedai Runcit Salmah"}}},
    {"text": "sediakan saya untuk lawatan ke Fresh Choice Mart", "catalogue": "admin", "lang": "ms", "expect": {"name": "brief_retailer", "args": {"retailer": "Fresh Choice Mart"}}},
    {"text": "ringkaskan Kedai Runcit Pak Din", "catalogue": "supervisor", "lang": "ms", "expect": {"name": "brief_retailer", "args": {"retailer": "Kedai Runcit Pak Din"}}},
    {"text": "open all orders", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "all_orders"}}},
    {"text": "show me orders", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "all_orders"}}},
    {"text": "go to the order list", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "all_orders"}}},
    {"text": "take me to the route planner", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "route_planner"}}},
    {"text": "open demand forecast", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "prediction"}}},
    {"text": "i want to see the reports", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "reports"}}},
    {"text": "pull up notifications", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "notifications"}}},
    {"text": "where can i find the retailers", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "retailer"}}},
    {"text": "open the audit log", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "audit_log"}}},
    {"text": "open team orders", "catalogue": "supervisor", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "team_orders"}}},
    {"text": "where is my team right now", "catalogue": "supervisor", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "team_activity"}}},
    {"text": "buka laporan", "catalogue": "admin", "lang": "ms", "expect": {"name": "navigate_to_page", "args": {"page": "reports"}}},
    {"text": "pergi ke notifikasi", "catalogue": "admin", "lang": "ms", "expect": {"name": "navigate_to_page", "args": {"page": "notifications"}}},
    {"text": "which shops did the reps go to", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "visit_logs"}}},
    {"text": "can i trust the forecast", "catalogue": "admin", "lang": "en", "expect": {"name": "navigate_to_page", "args": {"page": "trust_impact"}}},
    {"text": "approve order 5521", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "reject order 4410 please", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "approve everything that is submitted", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "save this route as a draft", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "prepare Hafizah's route for next week with six stops a day", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "publish the weekly route", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "delete all rejected orders", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "cancel order 3307", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "email these results to my manager", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "change Gopal's role to supervisor", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "luluskan pesanan 5521", "catalogue": "admin", "lang": "ms", "expect": null},
    {"text": "padam semua pesanan ditolak", "catalogue": "admin", "lang": "ms", "expect": null},
    {"text": "simpan laluan Hafizah sebagai draf", "catalogue": "admin", "lang": "ms", "expect": null},
    {"text": "hello", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "what is the weather today", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "tell me a joke", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "who won the match last night", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "apa khabar", "catalogue": "admin", "lang": "ms", "expect": null},
    {"text": "open orders and then open reports", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "show Hafizah's orders and export them", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "show payroll for this month", "catalogue": "admin", "lang": "en", "expect": null},
    {"text": "export all of these", "catalogue": "admin", "lang": "en", "actions": ["search_orders", "daily_briefing"], "expect": null},
    {"text": "can i have my briefing for today", "catalogue": "admin", "lang": "en", "actions": ["search_orders", "export_results"], "expect": null},
    {"text": "show Gopal's orders today", "catalogue": "admin", "lang": "en", "actions": ["daily_briefing"], "expect": null},
    {"text": "brief me on Kedai Bestari", "catalogue": "admin", "lang": "en", "actions": ["search_orders"], "expect": null},
    {"text": "explain the second item", "catalogue": "admin", "lang": "en", "actions": ["daily_briefing", "search_orders"], "expect": null},
    {"text": "mark it as in review", "catalogue": "admin", "lang": "en", "actions": ["daily_briefing", "explain_item"], "expect": null},
    {"text": "tolong eksport senarai ini", "catalogue": "admin", "lang": "ms", "actions": ["search_orders"], "expect": null},
    {"text": "tunjuk pesanan Gopal hari ini", "catalogue": "admin", "lang": "ms", "actions": [], "expect": null},
    {"text": "show Gopal's orders today", "catalogue": "admin", "lang": "en", "actions": ["search_orders", "export_results"], "expect": {"name": "search_orders", "args": {"salesperson": "Gopal", "period": "today"}}},
    {"text": "export all of these", "catalogue": "supervisor", "lang": "en", "actions": ["search_orders", "export_results"], "expect": {"name": "export_results", "args": {}}},
    {"text": "open reports", "catalogue": "admin", "lang": "en", "actions": [], "expect": {"name": "navigate_to_page", "args": {"page": "reports"}}},
    {"text": "buka laporan", "catalogue": "supervisor", "lang": "ms", "actions": ["daily_briefing"], "expect": null}
  ]
}


In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/train/make_dataset.py
from __future__ import annotations

import hashlib
import json
import random
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent
DATA = HERE / "data"
EVAL = HERE.parent / "eval_phrases.json"
SEED = 20260930
SIZES = {"train": 3600, "validation": 500, "test": 700}
NO_MATCH = "no_match"

CATALOGUE: dict[str, tuple[str, str, list[str], list[str]]] = {
    "dashboard": ("Dashboard", "Dashboard", ["home", "kpi", "summary", "action centre"], ["overview", "main dashboard", "go home"]),
    "salesman_activity": ("Salesman Tracking", "Salesman Tracking", ["gps", "map", "location", "live", "performance"], ["sales activity", "salesman tracking"]),
    "visit_logs": ("Visit Logs", "Visit Logs", ["visits", "history", "arrival", "follow-up"], ["visit history"]),
    "route_planner": ("Route Planner", "Weekly Route Planner", ["route", "optimisation", "estimate", "plan", "visit"], ["routes", "weekly routes", "route planning"]),
    "notifications": ("Notifications", "Notifications", ["alerts", "feed"], ["notification feed"]),
    "all_orders": ("All Orders", "All Orders", ["order approval", "sales order", "approve", "reject"], ["orders", "submitted orders"]),
    "purchase_order": ("Purchase Orders", "Purchase Orders", ["po", "procurement", "supply"], ["purchase orders"]),
    "special_requests": ("Special Requests", "Special Requests", ["exceptions", "approval"], ["request approvals"]),
    "reports": ("Reports", "Reports", ["analytics", "monthly trend", "product report", "monthly summary"], ["monthly analytics"]),
    "prediction": ("Prediction", "Demand Forecast", ["forecast", "sarimax", "demand", "restock recommendations"], ["demand forecast", "predictions"]),
    "trust_impact": ("Trust & Impact", "Trust & Impact Lab", ["accuracy", "impact", "evidence", "abstention", "trust"], ["trust impact", "abstentions"]),
    "department": ("Departments", "Departments", [], []),
    "category": ("Categories", "Categories", [], []),
    "product": ("Products", "Products", [], ["product catalogue", "catalogue"]),
    "retailer_scoresheets": ("Score Sheets", "Score Sheets", [], []),
    "users": ("Role Assignment", "Role Assignment", ["user management", "roles", "accounts"], []),
    "permissions": ("Role Permissions", "Role Permissions", ["permission", "matrix", "access", "authorization"], []),
    "user_directory": ("Users", "Users", [], ["staff directory"]),
    "branches": ("Branches", "Branch Management", [], ["branch management"]),
    "retailer": ("Retailers", "Retailers", [], ["outlets"]),
    "coordinate_review": ("Coordinate Review", "Coordinate Review", ["gps", "latitude", "longitude", "geocode", "map"], []),
    "import_preview": ("Import Centre", "Import Preview Centre", ["csv", "excel", "products", "retailers", "forecast"], []),
    "department_pic": ("Retailer Department PIC", "Retailer Department PIC", [], []),
    "terms_conditions": ("Terms & Conditions", "Terms & Conditions", [], []),
    "privacy_policy": ("Privacy Policy", "Privacy Policy", [], []),
    "backup": ("Backup", "Backup & Restore", [], []),
    "system_governance": ("System Governance", "System Governance", [], []),
    "system_health": ("System Health", "System Health", ["diagnostics", "data quality", "health", "checks"], []),
    "audit_log": ("Audit Log", "Audit Log", ["history", "changes", "actor", "governance"], []),
    "team_orders": ("Team Orders", "Team Orders", [], []),
    "team_visits": ("Team Visits", "Team Visits", [], []),
    "team_activity": ("Team Activity", "Team Activity", [], ["team map"]),
    "team_forecast": ("Team Forecast", "Team Forecast", [], []),
    "team_action_center": ("Action Centre", "Team Action Centre", [], ["team actions"]),
    "team_ai_summary": ("Weekly AI Summary", "Weekly AI Summary", [], ["team summary"]),
}
ORDER = list(CATALOGUE)
ROLES = {
    "branch_manager": ["dashboard", "salesman_activity", "visit_logs", "route_planner", "notifications", "all_orders", "purchase_order", "special_requests", "reports", "prediction"],
    "supervisor": ["dashboard", "notifications", "team_orders", "team_visits", "team_activity", "team_forecast", "team_action_center", "team_ai_summary"],
    "manager": ["dashboard", "salesman_activity", "visit_logs", "route_planner", "notifications", "all_orders", "special_requests", "reports", "prediction", "product", "retailer", "retailer_scoresheets", "department", "category"],
    "staff_admin": ["dashboard", "notifications", "user_directory", "users", "permissions", "branches", "audit_log", "privacy_policy", "terms_conditions", "department_pic"],
}
SIBLINGS = [
    ["all_orders", "purchase_order", "special_requests", "team_orders"],
    ["users", "permissions", "user_directory", "department_pic"],
    ["retailer", "retailer_scoresheets", "department_pic", "coordinate_review", "branches"],
    ["salesman_activity", "visit_logs", "route_planner", "team_activity", "team_visits"],
    ["prediction", "trust_impact", "team_forecast", "reports"],
    ["dashboard", "reports", "notifications", "team_action_center", "team_ai_summary"],
    ["system_health", "system_governance", "audit_log", "backup"],
    ["product", "category", "department", "import_preview"],
    ["privacy_policy", "terms_conditions"],
]

NOUNS: dict[str, list[str]] = {
    "dashboard": ["dashboard", "home page", "main overview", "kpi summary", "home screen", "landing page", "front page", "headline numbers"],
    "salesman_activity": ["salesman tracking", "sales rep map", "live map of the reps", "field team locations", "sales activity map", "rep gps tracking", "where the reps are", "live rep positions"],
    "visit_logs": ["visit logs", "visit history", "store visit records", "list of visits", "arrival records", "shop visit log", "follow-up visits", "visit records"],
    "route_planner": ["route planner", "weekly routes", "route planning", "the route plan for the week", "route optimiser", "planned routes", "weekly route schedule", "route scheduler"],
    "notifications": ["notifications", "alerts", "notification feed", "alert feed", "new messages from the system", "unread alerts", "system alerts", "alert inbox"],
    "all_orders": ["all orders", "orders", "submitted orders", "sales orders", "order approval queue", "customer orders", "order list", "pending orders"],
    "purchase_order": ["purchase orders", "procurement", "supply orders", "purchase order list", "po list", "supplier orders", "procurement orders", "purchasing"],
    "special_requests": ["special requests", "exceptions", "request approvals", "exception requests", "special order requests", "pending exceptions", "requests waiting approval", "retailer exception requests"],
    "reports": ["reports", "analytics", "monthly analytics", "monthly summary report", "sales reports", "reporting hub", "monthly reports", "performance reports"],
    "prediction": ["prediction", "demand forecast", "forecast page", "sales predictions", "forecasting", "restock recommendations", "predicted demand", "demand outlook"],
    "trust_impact": ["trust and impact", "forecast accuracy", "trust impact lab", "model evidence", "abstentions", "how reliable the forecasts are", "prediction trustworthiness", "forecast evidence"],
    "department": ["departments", "department list", "sales departments", "department settings", "department management", "manage departments"],
    "category": ["categories", "product categories", "category tree", "category hierarchy", "category management", "manage categories"],
    "product": ["products", "product catalogue", "catalogue", "product list", "product management", "sku list", "item catalogue", "product master"],
    "retailer_scoresheets": ["score sheets", "retailer score sheets", "scoresheets", "product score sheet for a retailer", "retailer scorecards", "score sheet settings"],
    "users": ["role assignment", "user management", "roles", "accounts", "assign roles", "user roles", "account roles", "role manager"],
    "permissions": ["role permissions", "permissions", "permission matrix", "access rights", "access control matrix", "authorization settings", "what each role can access", "role access settings"],
    "user_directory": ["users", "staff directory", "staff list", "user directory", "team members", "employee directory", "list of staff", "people directory"],
    "branches": ["branches", "branch management", "branch list", "branch locations", "manage branches", "our branches", "branch offices", "branch settings"],
    "retailer": ["retailers", "outlets", "retailer list", "shops", "retailer management", "outlet list", "customer stores", "retail outlets"],
    "coordinate_review": ["coordinate review", "gps coordinates", "outlet coordinates", "location review", "geocode review", "latitude and longitude checks", "map coordinates of outlets", "coordinate verification"],
    "import_preview": ["import centre", "import preview", "csv import", "excel import", "data import", "bulk upload", "import tool", "file upload"],
    "department_pic": ["department pic", "retailer department pic", "person in charge of departments", "department contacts", "pic assignment", "department person in charge"],
    "terms_conditions": ["terms and conditions", "terms of use", "app terms", "t and c", "user terms", "terms page"],
    "privacy_policy": ["privacy policy", "privacy notice", "data privacy statement", "privacy page", "privacy terms", "privacy info"],
    "backup": ["backup", "backup and restore", "data backup", "system backup", "backups", "export and restore"],
    "system_governance": ["system governance", "governance settings", "gps policy settings", "governance controls", "developer controls", "purge export controls"],
    "system_health": ["system health", "diagnostics", "data quality checks", "health checks", "system status", "system diagnostics", "platform health", "health monitor"],
    "audit_log": ["audit log", "audit trail", "change log", "activity log of changes", "governance history", "who changed what", "audit history", "record of changes"],
    "team_orders": ["team orders", "my team's orders", "orders from my team", "team order list", "team sales orders", "orders of my reps"],
    "team_visits": ["team visits", "my team's visits", "visits by my team", "team visit list", "team visit history", "visits by my reps"],
    "team_activity": ["team activity", "team map", "my team on the map", "team locations", "team tracking", "where my team is"],
    "team_forecast": ["team forecast", "my team's forecast", "forecast for my team", "team demand forecast", "team predictions", "team outlook"],
    "team_action_center": ["action centre", "team actions", "team action centre", "action items for my team", "things my team needs to do", "team to-do list"],
    "team_ai_summary": ["weekly ai summary", "team summary", "weekly summary of my team", "ai weekly digest", "team weekly ai summary", "this week's ai summary"],
}

INTENTS: dict[str, list[str]] = {
    "dashboard": ["what needs my attention today", "take me back to the start", "i want the big picture", "show me the headline numbers", "give me the daily overview", "back to the main screen", "what is going on today", "let me see the summary of everything", "go back to the beginning", "how is the business doing overall", "show the main kpis", "i want the home view"],
    "salesman_activity": ["how are my reps doing today", "where is everyone in the field right now", "are my salespeople out on the road", "i want to follow my field team live", "show me who is working right now", "which reps are active at the moment", "let me watch the sales team move", "see the location of the salesmen", "who is out visiting shops", "track my sales team", "how is the field team performing", "show gps positions of the salesmen"],
    "visit_logs": ["which shops did the reps go to", "did anyone visit the shop yesterday", "show me the history of shop visits", "when did the rep arrive at the store", "i want to check what visits happened last week", "list the store visits", "which outlets got visited", "look up past visits to a retailer", "who has been to which outlet", "show me the follow ups from visits", "how often are the stores being visited", "review the visits our people made"],
    "route_planner": ["plan where the reps go next week", "i want to schedule the sales trips", "set up the weekly visit plan", "arrange the journeys for my salesmen", "build the route for the week", "draft the visiting schedule", "who goes where on monday", "let me assign stores to reps for the week", "optimise the trips of the reps", "prepare the plan for the coming week", "publish the weekly plan of visits", "change the visiting route of a rep"],
    "notifications": ["did anything come up", "any new alerts for me", "what has been flagged", "i want to see my alerts", "is there anything new to read", "check what the system has told me", "anything urgent popped up", "show me what i missed", "read my unread alerts", "what warnings are there", "see the latest system messages", "review recent alerts"],
    "all_orders": ["which orders need my approval", "let me see the orders that came in", "i have orders waiting to be approved", "check the latest customer orders", "what did the shops order", "i want to look at the order list", "review the submitted orders", "how many orders came in today", "what orders are pending", "see the orders the reps submitted", "find an order from a retailer", "i need to approve or reject some orders"],
    "purchase_order": ["what are we buying from suppliers", "i need to order stock from the supplier", "let me raise a purchase order", "show me what we have ordered from vendors", "i want to restock from the supplier", "check the procurement list", "create an order to the supplier", "review what we ordered upstream", "how much are we buying from the vendor", "list the buying orders sent to suppliers", "see supply orders", "what did we procure this month"],
    "special_requests": ["a shop asked for something out of the ordinary", "any exceptions waiting for me", "someone requested an exception", "look at the unusual requests from retailers", "which special requests need a decision", "the exceptions need my approval", "check requests that fall outside policy", "requests that need my sign off", "are there any one-off requests", "see what the reps asked for as an exception", "review the off-policy requests", "show what is waiting for approval besides orders"],
    "reports": ["i need the monthly numbers", "how did we do this month", "show me the sales analytics", "let me look at the performance figures", "i want charts of the monthly sales", "generate an analysis of last month", "top products and top retailers", "i want to see the month in review", "compare this month to last month", "open the analysis section", "give me the sales statistics", "where can i find the monthly trend"],
    "prediction": ["how many units will we sell next week", "what will demand look like", "what should we restock", "i want to know the expected sales", "will this product sell more next month", "look at the sales outlook", "what is the projected demand", "which items will run short", "forecast the sales for the coming weeks", "predict what the shops will order", "what are we expecting to sell", "check the festive demand forecast"],
    "trust_impact": ["can i trust the forecast", "how good are the predictions", "how often is the model right", "is the forecast reliable", "show evidence the forecasts work", "when does the model refuse to predict", "how accurate has the model been", "i want proof the forecast helps", "check the forecasting quality", "did the forecasts hold up", "what is the model's track record", "does the model know when it is unsure"],
    "department": ["i want to edit the sales departments", "manage the departments we sell in", "add a new department", "change the department names", "which departments exist", "maintain the department list"],
    "category": ["i want to edit product categories", "organise the category hierarchy", "add a new product category", "which categories do we have", "restructure the product groups", "maintain the category tree"],
    "product": ["i want to edit an item", "let me look up a product", "what do we sell", "add a new sku", "check the price of an item", "open the list of things we sell", "update a product record", "find a product by name", "how many products are there", "review the item master"],
    "retailer_scoresheets": ["which products show for each retailer", "control what products a shop can see", "edit the product list for one shop", "configure the scoring sheet of a retailer", "set the products on a retailer sheet", "manage what each store scores"],
    "users": ["change someone's role", "i need to assign roles to people", "promote a user to manager", "make this person a supervisor", "give a staff member a new role", "who has which role", "set the role of a new hire", "manage user accounts and their roles", "switch a user from salesman to admin", "grant a role to an employee"],
    "permissions": ["what is each role allowed to do", "who can access which page", "change what a role can see", "edit access rules for roles", "check which permissions managers have", "set what supervisors are allowed to open", "adjust the access matrix", "control what each role is allowed", "which role can approve orders", "review role privileges"],
    "user_directory": ["who works here", "find a staff member", "look up a colleague's contact", "who are the people in the system", "list everyone with an account", "see the staff and their contact details", "which users exist", "check who is on the team", "find the phone number of an employee", "browse the employee records"],
    "branches": ["where are our branch offices", "add a new branch", "edit the branch address", "how many branches are there", "manage the physical locations of the company", "list of our branch sites", "update a branch location", "which branches do we have"],
    "retailer": ["which shops do we sell to", "add a new retailer", "find a shop by name", "how many outlets do we have", "edit a store's details", "look up a customer outlet", "browse the retailer records", "show the stores we serve", "update the address of a shop", "our list of customers"],
    "coordinate_review": ["are the shop locations correct on the map", "verify where the outlets are", "check the outlet pins that need review", "fix the location of a shop", "which outlets have doubtful locations", "confirm the gps of stores", "review imported locations", "i want to validate outlet positions", "look at unverified store coordinates", "approve or reject candidate locations"],
    "import_preview": ["i want to upload a spreadsheet", "bring in data from a file", "load some products from excel", "upload the retailer list", "import my csv file", "preview a file before importing", "add many rows from a file at once", "put in the sales data from a sheet"],
    "department_pic": ["who is in charge of a department at a shop", "assign a person to a department at a retailer", "set the contact for a department at a store", "which pic looks after which department", "change the person responsible for a retailer's department", "edit department contacts per shop"],
    "terms_conditions": ["what are the rules of using the app", "read the legal terms", "i want the terms of the service", "where are the conditions of use", "show the agreement users accept", "look at the legal conditions"],
    "privacy_policy": ["how is personal data handled", "what is the data privacy statement", "read how you use our information", "where is the privacy notice", "show me how data is protected", "i want the policy about personal data"],
    "backup": ["save a copy of all the data", "i want to restore from a saved copy", "export everything as a safety copy", "make an archive of the system", "recover data from an earlier backup", "download a full data export"],
    "system_governance": ["change the gps policy", "developer only controls", "manage system level access rules", "export data before purging it", "open the developer governance settings", "configure the system controls for developers"],
    "system_health": ["is everything working properly", "run a check on the system", "are there data quality problems", "is the platform up", "any problems with the data", "diagnose the system", "how healthy is the data", "look for broken records", "check that all services are ok", "are there any errors in the system"],
    "audit_log": ["who changed this record", "what happened to the data recently", "see who edited what", "i need a trail of changes", "who did what and when", "show me the history of admin actions", "look into who modified the settings", "check the record of user actions", "trace a change back to a person", "review admin activity history"],
    "team_orders": ["which orders did my reps submit", "orders from the people i supervise", "check what my team sold", "let me see my team's order list", "what did my salesmen order today", "review orders by my team"],
    "team_visits": ["which shops has my team visited", "did my reps do their visits", "the visits my team made", "check my team's store calls", "how many visits did my people do", "review my team's shop visits"],
    "team_activity": ["where is my team right now", "how is my team doing today", "follow my team on the map", "are my reps out on the road", "watch my team move", "who in my team is active"],
    "team_forecast": ["what will my team sell next week", "expected demand for my area", "predicted sales for my reps", "what should my team push", "the outlook for my team's territory", "will my team hit the sales target"],
    "team_action_center": ["what does my team need to do", "the tasks waiting for my team", "which actions are open for my reps", "what should i follow up with my team", "what needs fixing in my area", "list the pending tasks for my people"],
    "team_ai_summary": ["give me a digest of my team's week", "summarise how my team did this week", "the ai's recap of my team", "a written summary of the week for my reps", "what did the ai say about my team's week", "recap the week for my team"],
}

LEADS_TRAIN = [
    "{n}", "open {n}", "show me {n}", "go to {n}", "take me to {n}", "i want to see {n}", "pull up {n}", "let me see {n}",
    "can you open {n}", "navigate to {n}", "switch to {n}", "i need {n}", "i would like to view {n}", "check {n}",
    "{n} please", "please open {n}", "i wanna look at {n}", "bring up {n}", "display {n}", "buka {n}", "pergi ke {n}",
    "could you show me {n}", "let's go to {n}", "get me to {n}", "hey open {n}", "where can i find {n}",
]
LEADS_VAL = ["jump to {n}", "i'd like to check {n}", "can i see {n}", "{n} now", "head to {n}"]
LEADS_TEST = ["load {n}", "look at {n}", "tunjuk {n}", "would you mind showing me {n}", "i'm trying to get to {n}", "show {n}", "view {n}", "{n} page", "take me over to {n}"]
FILLERS = ["um", "uh", "hey", "ok", "so", "right", "yeah", "hmm", "please", "quickly", "okay so"]

CHAT = ["hello", "hi there", "good morning", "how are you", "thank you", "thanks a lot", "who are you", "what can you do", "tell me a joke", "what is the weather today", "what time is it", "sing me a song", "banana", "um", "nothing", "never mind", "stop", "cancel that", "how do i cook rice", "who won the football match", "what is the capital of france", "play some music", "call my manager", "turn off the lights", "set an alarm for seven", "remind me to buy milk", "i am hungry", "it is so hot today", "tell me something interesting", "what should i eat for lunch", "translate good morning into malay", "how far is the moon", "what is two plus two", "are you a robot", "good night", "see you tomorrow", "how is my day going", "what is the meaning of life", "open the window", "what is your name", "this is boring", "help", "can you hear me", "testing one two three", "is it raining outside", "book me a flight", "text my wife", "how tall is mount kinabalu", "read me the news", "i love this song"]
OUT_OF_DOMAIN = ["the weather forecast for tomorrow", "the football scores", "a cooking recipe", "the latest news", "my horoscope", "traffic to the airport", "the exchange rate", "a funny video", "stock market prices", "my calendar", "my email inbox", "the hotel bookings", "a map of kuching", "the cinema times", "some music"]
UNAVAILABLE = ["inventory levels", "warehouse stock", "payroll", "employee salaries", "invoices", "customer complaints", "supplier contracts", "expense claims", "marketing campaigns", "loyalty points", "the bank balance", "attendance records", "leave requests", "delivery trucks", "the accounting ledger", "tax filings", "the crm contacts", "shift rosters", "the returns desk", "credit limits"]
ACTIONS_TRAIN = ["delete", "approve", "reject", "publish", "remove", "cancel", "restore", "reset", "deactivate", "clear", "wipe"]
ACTIONS_VAL = ["erase", "revoke"]
ACTIONS_TEST = ["purge", "drop", "disable", "confirm all", "submit"]
ACTION_OBJECTS = ["all the orders", "this user", "the route", "the weekly route plan", "everyone's password", "the backup", "that retailer", "the special request", "all pending requests", "order 1234", "purchase order 88", "the forecast", "the visit logs", "the product", "this branch", "the audit log", "notifications", "the role permissions", "the retailer coordinates", "the department"]
SPECIFIC_ACTIONS = ["send an email to the supplier", "email the retailer", "print the report", "call the salesman", "message my team", "share the dashboard with the ceo", "change my password", "log me out"]
CONNECT_TRAIN = ["{a} and then {b}", "{a} then {b}", "open {a} and {b}", "go to {a} and after that {b}", "show {a} and also open {b}", "{a} followed by {b}"]
CONNECT_VAL = ["first {a} then {b}", "{a} and {b} together"]
CONNECT_TEST = ["show {a} and check {b}", "take me to {a} and then to {b}", "{a}, then {b}", "{a} plus {b}"]


def alias_pool(key: str) -> list[str]:
    label, title, kws, extras = CATALOGUE[key]
    out: list[str] = []
    for a in [label, title, key.replace("_", " "), *kws, *extras]:
        if a not in out:
            out.append(a)
    return out


def aliases_for(key: str, seed: int) -> list[str]:
    pool = alias_pool(key)
    if seed == 0:
        return pool[:8]
    r = random.Random(f"{key}:{seed}")
    k = r.randint(1, min(8, len(pool)))
    return r.sample(pool, k)


def request_pages(row: dict) -> list[dict]:
    return [
        {"key": k, "label": CATALOGUE[k][0], "aliases": aliases_for(k, row["aliasSeed"])}
        for k in row["pages"]
    ]


def norm(text: str) -> str:
    return " ".join(text.lower().split())


def split_nouns(key: str, split: str) -> list[str]:
    nouns = NOUNS[key]
    cut = max(1, len(nouns) - max(1, len(nouns) // 4))
    return nouns[:cut] if split == "train" else nouns


def split_intents(key: str, split: str) -> list[str]:
    want = {"train": lambda i: i % 6 not in (2, 5), "validation": lambda i: i % 6 == 2, "test": lambda i: i % 6 == 5}[split]
    picked = [s for i, s in enumerate(INTENTS[key]) if want(i)]
    return picked or INTENTS[key][:1]


def typo(word: str, r: random.Random) -> str:
    if len(word) < 5:
        return word
    i = r.randrange(1, len(word) - 2)
    op = r.choice(["swap", "drop", "dup", "sub"])
    if op == "swap":
        return word[:i] + word[i + 1] + word[i] + word[i + 2:]
    if op == "drop":
        return word[:i] + word[i + 1:]
    if op == "dup":
        return word[:i] + word[i] + word[i:]
    return word[:i] + r.choice("aeiounrst") + word[i + 1:]


def maybe_typo(phrase: str, r: random.Random, p: float = 0.12) -> str:
    if r.random() >= p:
        return phrase
    words = phrase.split()
    idx = r.randrange(len(words))
    words[idx] = typo(words[idx], r)
    return " ".join(words)


def style(text: str, r: random.Random) -> str:
    if r.random() < 0.10:
        text = f"{r.choice(FILLERS)} {text}"
    x = r.random()
    if x < 0.08:
        text = text.capitalize()
    elif x < 0.12:
        text = text.upper()
    y = r.random()
    if y < 0.06:
        text += "."
    elif y < 0.09:
        text += "?"
    elif y < 0.10:
        text += "!"
    return text


def pick_pages(r: random.Random, target: str | None, include_target: bool) -> list[str]:
    mode = r.random()
    if mode < 0.32:
        admin = [k for k in ORDER if not k.startswith("team_")]
        pages = admin if r.random() < 0.7 else [k for k in admin if r.random() < 0.7]
    elif mode < 0.62:
        pages = list(r.choice(list(ROLES.values())))
    else:
        pages = r.sample(ORDER, r.randint(3, 16))
    if target is not None:
        sib = next((g for g in SIBLINGS if target in g), [])
        if include_target:
            if target not in pages:
                pages.append(target)
        else:
            pages = [p for p in pages if p != target]
            if r.random() < 0.5:
                pages += [s for s in sib if s != target and s not in pages][:3]
            if len(pages) < 3:
                pages += [p for p in r.sample(ORDER, 4) if p != target and p not in pages]
    pages = [p for p in dict.fromkeys(pages)]
    if r.random() < 0.75:
        pages.sort(key=ORDER.index)
    else:
        r.shuffle(pages)
    return pages


def row(text: str, pages: list[str], target: str, kind: str, family: str, r: random.Random) -> dict:
    return {
        "text": " ".join(text.split()),
        "pages": pages,
        "aliasSeed": 0 if r.random() < 0.6 else r.randint(1, 9999),
        "target": target,
        "kind": kind,
        "family": family,
    }


def det(r: random.Random) -> str:
    return r.choice(["", "", "the ", "my "])


def make_noun_phrase(lead: str, noun: str, r: random.Random, d: str) -> str:
    n = maybe_typo(noun, r)
    if lead.startswith(("buka", "tunjuk", "pergi")) or lead == "{n}" or lead == "{n} please":
        d = ""
    return lead.format(n=d + n)


def gen_row(kind: str, split: str, r: random.Random) -> dict:
    leads = {"train": LEADS_TRAIN, "validation": LEADS_VAL, "test": LEADS_TEST}[split]
    if kind in ("noun", "gated_noun"):
        key = r.choice(ORDER)
        lead = r.choice(leads)
        text = make_noun_phrase(lead, r.choice(split_nouns(key, split)), r, det(r))
        gated = kind == "gated_noun"
        return row(style(text, r), pick_pages(r, key, not gated), NO_MATCH if gated else key, kind, "lead:" + lead, r)
    if kind in ("intent", "gated_intent"):
        key = r.choice(ORDER)
        sentence = r.choice(split_intents(key, split))
        gated = kind == "gated_intent"
        return row(style(maybe_typo(sentence, r, 0.06), r), pick_pages(r, key, not gated), NO_MATCH if gated else key, kind, "intent:" + sentence, r)
    if kind == "chat":
        pool = CHAT
        want = {"train": lambda i: i % 6 not in (2, 5), "validation": lambda i: i % 6 == 2, "test": lambda i: i % 6 == 5}[split]
        sentence = r.choice([s for i, s in enumerate(pool) if want(i)])
        return row(style(sentence, r), pick_pages(r, None, True), NO_MATCH, kind, "chat:" + sentence, r)
    if kind == "out_of_domain":
        lead = r.choice(leads)
        pool = OUT_OF_DOMAIN if r.random() < 0.5 else UNAVAILABLE
        text = make_noun_phrase(lead, r.choice(pool), r, det(r))
        return row(style(text, r), pick_pages(r, None, True), NO_MATCH, kind, "lead:" + lead, r)
    if kind == "action":
        verbs = {"train": ACTIONS_TRAIN, "validation": ACTIONS_VAL, "test": ACTIONS_TEST}[split]
        if r.random() < 0.15:
            text, verb = r.choice(SPECIFIC_ACTIONS), "specific"
        else:
            verb = r.choice(verbs)
            text = f"{verb} {r.choice(ACTION_OBJECTS)}"
            if r.random() < 0.25:
                text = f"{r.choice(['please', 'can you', 'go ahead and'])} {text}"
        key = r.choice(ORDER)
        return row(style(text, r), pick_pages(r, key, True), NO_MATCH, kind, "action:" + verb, r)
    if kind == "compound":
        conn = r.choice({"train": CONNECT_TRAIN, "validation": CONNECT_VAL, "test": CONNECT_TEST}[split])
        a, b = r.sample(ORDER, 2)
        text = conn.format(a=r.choice(split_nouns(a, split)), b=r.choice(split_nouns(b, split)))
        pages = pick_pages(r, a, True)
        if b not in pages:
            pages.append(b)
        return row(style(text, r), pages, NO_MATCH, kind, "connect:" + conn, r)
    raise ValueError(kind)


MIX = [("noun", 0.30), ("intent", 0.22), ("gated_noun", 0.10), ("gated_intent", 0.05), ("chat", 0.06),
       ("out_of_domain", 0.13), ("action", 0.09), ("compound", 0.05)]


def build(split: str, n: int, seed: int, banned: set[str]) -> list[dict]:
    r = random.Random(f"{seed}:{split}")
    kinds = [k for k, _ in MIX]
    weights = [w for _, w in MIX]
    out: list[dict] = []
    seen: set[tuple[str, tuple[str, ...]]] = set()
    guard = 0
    while len(out) < n and guard < n * 50:
        guard += 1
        rw = gen_row(r.choices(kinds, weights)[0], split, r)
        sig = (norm(rw["text"]), tuple(rw["pages"]))
        if norm(rw["text"]) in banned or sig in seen:
            continue
        seen.add(sig)
        out.append(rw)
    return out


def main() -> None:
    eval_texts = {norm(c["text"]) for c in json.loads(EVAL.read_text(encoding="utf-8"))["cases"]}
    test = build("test", SIZES["test"], SEED, eval_texts)
    test_texts = {norm(x["text"]) for x in test}
    val = build("validation", SIZES["validation"], SEED, eval_texts | test_texts)
    val_texts = {norm(x["text"]) for x in val}
    train = build("train", SIZES["train"], SEED, eval_texts | test_texts | val_texts)
    train_texts = {norm(x["text"]) for x in train}
    for name, held in (("eval_phrases.json", eval_texts), ("validation", val_texts), ("test", test_texts)):
        overlap = train_texts & held
        if overlap:
            sys.exit(f"train overlaps {name}: {sorted(overlap)[:5]}")
    if test_texts & val_texts or test_texts & eval_texts or val_texts & eval_texts:
        sys.exit("held-out sets overlap")
    for name, rows in (("train", train), ("validation", val), ("test", test)):
        DATA.mkdir(parents=True, exist_ok=True)
        path = DATA / f"{name}.jsonl"
        path.write_text("".join(json.dumps(x, ensure_ascii=False) + "\n" for x in rows), encoding="utf-8")
        kinds: dict[str, int] = {}
        for x in rows:
            kinds[x["kind"]] = kinds.get(x["kind"], 0) + 1
        nm = sum(x["target"] == NO_MATCH for x in rows)
        digest = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
        print(f"{name}: {len(rows)} rows, no_match={nm}, sha256[:16]={digest}, kinds={kinds}")
    print("OK: zero exact-text overlap between train and (validation, test, eval_phrases.json)")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/train/make_copilot_dataset.py
from __future__ import annotations

import argparse
import copy
import hashlib
import json
import random
import re
import sys
from pathlib import Path
from typing import Any

from tools.voice.dashboard_llm import copilot_schema as schema
from tools.voice.dashboard_llm.train import make_dataset as md

HERE = Path(__file__).resolve().parent
LLM_DIR = HERE.parent
DATA = HERE / "data_copilot"
FROZEN_FILES = ("copilot_eval_phrases.json", "eval_phrases.json", "fresh_phrases.json")
SEED = 20261006
SIZES = {"train": 8000, "validation": 800, "test": 1200}

NAV_INTENT_BLOCKLIST = {
    "what needs my attention today", "what is going on today", "give me the daily overview",
    "how many orders came in today", "what orders are pending", "check the latest customer orders",
    "review the submitted orders", "see the orders the reps submitted", "find an order from a retailer",
    "which orders need my approval", "i have orders waiting to be approved", "what did the shops order",
}
NAV_NOUN_BLOCKLIST = {"submitted orders", "pending orders"}

SALESPEOPLE = {
    "train": ["Ahmad", "Siti", "Wong", "Kumar", "Aisyah", "Daniel", "Farid", "Mei Ling", "Raj", "Hafiz",
              "Nurul", "Jason", "Priya", "Zul", "Amirul", "Lim", "Syafiq", "Devi", "Hana", "Ravi",
              "Fatimah", "Chong", "Irfan", "Ali", "Siew Lan", "Mohan", "Kamal", "Azlan", "Tan", "Suria"],
    "validation": ["Haziq", "Wei Jie", "Nadia", "Selvam", "Rina"],
    "test": ["Faizal", "Li Na", "Anand", "Syazwani", "Joseph", "Dewi", "Hakim", "Mei Yee"],
}
RETAILERS = {
    "train": ["Kedai Maju", "Pasar Mini Sentosa", "Mini Mart Sejahtera", "Pasaraya Jaya", "Kedai Runcit Ali",
              "Stesen Mart Bintang", "Everfresh Mart", "Gedung Rakyat", "Kedai Makmur", "Sri Murni Mart",
              "Borneo Grocer", "Lucky Mini Market", "Kedai Runcit Hasan", "Bintang Supermart", "Sin Hock Trading",
              "Pasar Raya Kenanga", "Kedai Mewah", "Hup Seng Store", "Mutiara Mart", "Green Valley Mart",
              "KK Mart Jalan Utama", "Kedai Sinar", "Pekan Baru Minimarket", "Berkat Mart", "Wawasan Supermarket"],
    "validation": ["Kedai Harapan", "Sentral Mart", "Cahaya Minimarket", "Teck Soon Trading"],
    "test": ["Kedai Bahagia", "Metro Grocer", "Pasaraya Dahlia", "Seng Huat Store", "Kedai Runcit Kassim", "Sunrise Mart"],
}

def _pool(base: dict[str, list[str]], held: dict[str, list[str]]) -> dict[str, dict[str, list[str]]]:
    keys = set(base) | set(held)
    return {k: {"train": base.get(k, []), "held": base.get(k, []) + held.get(k, [])} for k in keys}


PERIOD = {
    "en": _pool(
        {"today": ["today"], "yesterday": ["yesterday"], "this_week": ["this week"], "last_week": ["last week"],
         "this_month": ["this month"], "last_month": ["last month"]},
        {"today": ["so far today", "today only"], "yesterday": ["from yesterday"],
         "this_week": ["over this week", "in this week"], "last_week": ["during last week", "over last week"],
         "this_month": ["throughout this month"], "last_month": ["during last month", "over last month"]}),
    "ms": _pool(
        {"today": ["hari ini"], "yesterday": ["semalam"], "this_week": ["minggu ini"],
         "last_week": ["minggu lepas", "minggu lalu"], "this_month": ["bulan ini"],
         "last_month": ["bulan lepas", "bulan lalu"]},
        {"today": ["setakat hari ini"], "yesterday": ["pada semalam"], "this_week": ["sepanjang minggu ini"],
         "last_week": ["sepanjang minggu lepas"], "this_month": ["bagi bulan ini"],
         "last_month": ["sepanjang bulan lepas"]}),
}
ORDER_STATUS = {
    "en": _pool({"submitted": ["submitted", "new", "pending"], "approved": ["approved", "accepted"],
                 "rejected": ["rejected", "declined", "refused"]},
                {"approved": ["signed off"], "rejected": ["turned down", "denied"], "submitted": ["unprocessed"]}),
    "ms": _pool({"submitted": ["dihantar", "baharu"], "approved": ["diluluskan"], "rejected": ["ditolak"]},
                {"approved": ["telah diluluskan"], "rejected": ["telah ditolak"]}),
}
ITEM_STATUS = {
    "en": _pool({"in_review": ["in review", "under review"], "resolved": ["resolved", "done"], "ignored": ["ignored", "dismissed"]},
                {"in_review": ["pending review"], "resolved": ["completed"], "ignored": ["skipped"]}),
    "ms": _pool({"in_review": ["dalam semakan", "sedang disemak"], "resolved": ["selesai"], "ignored": ["diabaikan"]},
                {"in_review": ["untuk disemak"], "resolved": ["telah selesai"], "ignored": ["dilangkau"]}),
}
ORDINAL = {
    "en": ["first", "second", "third", "fourth", "fifth", "sixth", "seventh", "eighth", "ninth", "tenth"],
    "ms": ["pertama", "kedua", "ketiga", "keempat", "kelima", "keenam", "ketujuh", "kelapan", "kesembilan", "kesepuluh"],
}

SEARCH = {
    "train": [
        ("en", "show {sp}'s orders"), ("en", "show {sp}'s orders {pd}"), ("en", "show {sp}'s {st} orders {pd}"),
        ("en", "{st} orders from {rt}"), ("en", "orders from {rt} {pd}"), ("en", "list the orders {sp} submitted {pd}"),
        ("en", "how many orders did {sp} submit {pd}"), ("en", "find {st} orders for {rt} {pd}"),
        ("en", "show me orders {pd}"), ("en", "orders {pd}"), ("en", "what did {sp} sell {pd}"),
        ("en", "show {sp} orders for {rt}"), ("en", "get {sp}'s {st} orders at {rt}"), ("en", "list {st} orders {pd}"),
        ("en", "{sp} orders {pd}"), ("en", "pull up {rt} orders {pd}"), ("en", "any orders from {rt} {pd}"),
        ("en", "check {sp}'s orders {pd}"), ("en", "search orders by {sp}"), ("en", "search orders for {rt}"),
        ("en", "show me the {st} orders from {rt}"),
        ("ms", "tunjuk pesanan {sp} {pd}"), ("ms", "pesanan {st} dari {rt} {pd}"), ("ms", "senaraikan pesanan {sp} {pd}"),
        ("ms", "cari pesanan {rt}"), ("ms", "berapa pesanan {sp} {pd}"), ("ms", "papar pesanan {sp}"),
        ("ms", "pesanan {pd}"), ("ms", "tunjuk pesanan {st} {pd}"), ("ms", "pesanan {sp} di {rt} {pd}"),
        ("mix", "show pesanan {sp} {pd}"), ("mix", "tunjuk orders {rt} {pd}"), ("mix", "show orders {sp} {pd}"),
        ("en", "only {rt}"), ("en", "just {rt}"), ("en", "only the {st} ones"), ("en", "just the {st} ones"),
        ("en", "{pd} instead"), ("en", "for {sp}"), ("en", "only for {sp}"), ("en", "make it {pd}"),
        ("en", "same but for {rt}"), ("en", "filter by {st}"), ("en", "what about {sp}"), ("en", "how about {rt}"),
        ("ms", "hanya {rt}"), ("ms", "yang {st} sahaja"), ("ms", "untuk {sp} sahaja"), ("ms", "{pd} sahaja"),
        ("ms", "bagaimana dengan {sp}"),
    ],
    "validation": [
        ("en", "give me {sp}'s orders {pd}"), ("en", "which orders did {sp} submit {pd}"), ("en", "{st} orders for {rt}"),
        ("en", "find orders from {rt} {pd}"), ("en", "view {sp}'s {st} orders"), ("en", "orders by {sp} at {rt}"),
        ("en", "show every {st} order {pd}"), ("ms", "paparkan pesanan {sp} {pd}"), ("ms", "pesanan {st} untuk {rt}"),
        ("mix", "tunjuk {st} orders {sp}"),
        ("en", "limit it to {rt}"), ("en", "narrow it to the {st} ones"), ("en", "switch to {pd}"), ("en", "and for {sp}"),
        ("ms", "hadkan kepada {rt}"), ("ms", "tukar kepada {pd}"),
    ],
    "test": [
        ("en", "i need {sp}'s orders {pd}"), ("en", "has {rt} ordered {pd}"), ("en", "display {st} orders {pd}"),
        ("en", "look up {sp}'s {st} orders at {rt}"), ("en", "everything {sp} sold {pd}"),
        ("en", "orders placed by {sp} {pd}"), ("en", "{rt} {st} orders"), ("ms", "saya nak tengok pesanan {sp} {pd}"),
        ("ms", "ada pesanan dari {rt} {pd}"), ("ms", "lihat pesanan {st} {pd}"), ("mix", "check pesanan {rt} {pd}"),
        ("en", "restrict to {rt}"), ("en", "show only {st} ones"), ("en", "change it to {pd}"), ("en", "now {sp}'s"),
        ("ms", "tapis {rt}"), ("ms", "ubah kepada {pd}"), ("ms", "pula untuk {sp}"),
    ],
}
BRIEFING = {
    "train": [("en", "what needs my attention today"), ("en", "give me my daily briefing"),
              ("en", "what should i focus on today"), ("en", "brief me for today"), ("en", "what is on my plate today"),
              ("en", "any priorities for today"), ("en", "start my day"), ("en", "what are my priorities"),
              ("en", "daily briefing please"), ("ms", "apa perlu saya beri perhatian hari ini"),
              ("ms", "beri saya ringkasan harian"), ("ms", "apa keutamaan saya hari ini"), ("ms", "mulakan hari saya")],
    "validation": [("en", "what's urgent today"), ("en", "what should i tackle first"),
                   ("ms", "ringkasan hari ini"), ("ms", "apa yang mendesak")],
    "test": [("en", "what is urgent this morning"), ("en", "walk me through today"), ("en", "morning briefing"),
             ("ms", "apa yang mendesak hari ini"), ("ms", "apa saya patut buat dahulu hari ini")],
}
EXPORT = {
    "train": [("en", "export these"), ("en", "download this as csv"), ("en", "export the results"),
              ("en", "save these to a file"), ("en", "give me a csv of this"), ("en", "export to csv"),
              ("en", "export this list"), ("ms", "eksport ini"), ("ms", "muat turun senarai ini"),
              ("ms", "eksport ke csv"), ("ms", "simpan keputusan ini sebagai fail")],
    "validation": [("en", "download these results"), ("en", "get me a spreadsheet of this"), ("ms", "eksport senarai ini")],
    "test": [("en", "can i have this as a csv"), ("en", "export everything shown"), ("en", "save this list as a file"),
             ("ms", "muat turun sebagai csv"), ("ms", "boleh eksport yang ini")],
}
EXPLAIN = {
    "train": [("en", "explain the {ord} item"), ("en", "explain item {n}"), ("en", "why is the {ord} one there"),
              ("en", "tell me more about the {ord} item"), ("en", "what does the {ord} item mean"),
              ("en", "why is item {n} important"), ("en", "explain number {n}"), ("en", "explain this one"),
              ("en", "why is that important"), ("en", "why is it there"), ("en", "explain it"),
              ("en", "what is this about"), ("ms", "terangkan item {ord}"), ("ms", "kenapa item {ord} penting"),
              ("ms", "terangkan nombor {n}"), ("ms", "terangkan yang ini"), ("ms", "kenapa ini penting"),
              ("ms", "apa maksud item {n}")],
    "validation": [("en", "walk me through the {ord} item"), ("en", "why did the {ord} one come up"),
                   ("en", "explain that"), ("ms", "huraikan item {ord}"), ("ms", "kenapa item {n} muncul")],
    "test": [("en", "break down item {n}"), ("en", "what is the {ord} item about"), ("en", "say more about the {ord} one"),
             ("en", "why is this here"), ("ms", "jelaskan item {n}"), ("ms", "apa itu item {ord}"),
             ("ms", "kenapa yang ini penting")],
}
SET_STATUS = {
    "train": [("en", "mark the {ord} item as {sv}"), ("en", "mark it as {sv}"), ("en", "set item {n} to {sv}"),
              ("en", "set it to {sv}"), ("en", "mark item {n} as {sv}"), ("en", "move the {ord} one to {sv}"),
              ("en", "ignore the {ord} item", {"status": "ignored"}), ("en", "dismiss it", {"status": "ignored"}),
              ("en", "ignore item {n}", {"status": "ignored"}), ("ms", "tandakan item {ord} sebagai {sv}"),
              ("ms", "tandakan sebagai {sv}"), ("ms", "tetapkan item {n} kepada {sv}"),
              ("ms", "abaikan item {ord}", {"status": "ignored"}), ("ms", "abaikan yang ini", {"status": "ignored"})],
    "validation": [("en", "flag the {ord} item as {sv}"), ("en", "set the {ord} one to {sv}"),
                   ("en", "skip the {ord} item", {"status": "ignored"}), ("ms", "letakkan item {ord} sebagai {sv}")],
    "test": [("en", "update item {n} to {sv}"), ("en", "switch it to {sv}"), ("en", "mark number {n} as {sv}"),
             ("en", "dismiss item {n}", {"status": "ignored"}), ("ms", "kemas kini item {n} kepada {sv}"),
             ("ms", "abaikan item {n}", {"status": "ignored"})],
}
BRIEF = {
    "train": [("en", "brief me on {rt}"), ("en", "prepare me for my visit to {rt}"),
              ("en", "what should i discuss with {rt}"), ("en", "give me a visit brief for {rt}"),
              ("en", "summarise {rt} for me"), ("en", "visit brief {rt}"), ("en", "brief me before i see {rt}"),
              ("en", "what do i need to know about {rt}"), ("en", "brief me on this retailer"),
              ("en", "give me the visit brief"), ("ms", "beri taklimat tentang {rt}"),
              ("ms", "sediakan saya untuk lawatan ke {rt}"), ("ms", "ringkaskan {rt}"),
              ("ms", "apa yang perlu saya bincang dengan {rt}"), ("ms", "taklimat lawatan untuk {rt}"),
              ("ms", "beri taklimat tentang kedai ini")],
    "validation": [("en", "get me ready to visit {rt}"), ("en", "talk me through {rt}"),
                   ("ms", "bagi saya ringkasan {rt}"), ("en", "brief me on this shop")],
    "test": [("en", "i'm meeting {rt} later, what should i know"), ("en", "give me the background on {rt}"),
             ("ms", "ringkasan lawatan {rt}"), ("ms", "sebelum saya jumpa {rt}, apa perlu saya tahu"),
             ("en", "prepare me for this visit")],
}
WRITE_NEGATIVE = {
    "train": ["approve order {no}", "reject order {no}", "approve all submitted orders", "save this route as a draft",
              "prepare {sp}'s route for next week", "create a route for {sp}", "publish the route",
              "delete these orders", "cancel order {no}", "send these results to the supplier",
              "email this to my manager", "change {sp}'s role", "luluskan pesanan {no}", "tolak pesanan {no}",
              "padam semua pesanan", "simpan laluan sebagai draf", "sediakan laluan untuk {sp}",
              "luluskan semua pesanan"],
    "validation": ["approve the order for {rt}", "save the route as draft", "remove order {no}",
                   "set up a route for {sp} next week", "luluskan pesanan {rt}", "terbitkan laluan"],
    "test": ["go ahead and approve order {no}", "decline order {no}", "plan {sp}'s week and save it",
             "email me the orders", "batalkan pesanan {no}", "simpan draf laluan untuk {sp}"],
}

POLITE = {
    "train": ["can you", "could you", "please", "i want to", "tolong", "boleh", "saya nak", "i need to"],
    "validation": ["kindly", "would you", "boleh tak"],
    "test": ["hey can you", "i'd like to", "mohon", "tolong boleh"],
}

MS_NOUNS = {
    "dashboard": (["papan pemuka", "halaman utama"], ["skrin utama"]),
    "salesman_activity": (["penjejakan jurujual", "peta jurujual"], ["lokasi pasukan jualan"]),
    "visit_logs": (["log lawatan", "sejarah lawatan"], ["rekod lawatan"]),
    "route_planner": (["perancang laluan", "laluan mingguan"], ["pelan laluan"]),
    "notifications": (["notifikasi", "amaran"], ["pemberitahuan"]),
    "all_orders": (["pesanan", "semua pesanan", "senarai pesanan"], ["pesanan jualan"]),
    "purchase_order": (["pesanan pembelian", "pesanan belian"], ["pembelian pembekal"]),
    "special_requests": (["permintaan khas"], ["permintaan istimewa"]),
    "reports": (["laporan", "laporan bulanan"], ["analisis jualan"]),
    "prediction": (["ramalan permintaan", "ramalan"], ["unjuran permintaan"]),
    "product": (["produk", "katalog produk"], ["senarai produk"]),
    "retailer": (["peruncit", "senarai peruncit"], ["kedai pelanggan"]),
    "branches": (["cawangan"], ["senarai cawangan"]),
    "user_directory": (["senarai pengguna", "direktori kakitangan"], ["kakitangan"]),
    "users": (["tetapan peranan"], ["peranan pengguna"]),
    "audit_log": (["log audit"], ["jejak audit"]),
    "import_preview": (["pusat import"], ["muat naik fail"]),
    "backup": (["sandaran"], ["salinan sandaran"]),
    "team_orders": (["pesanan pasukan"], ["pesanan pasukan saya"]),
    "team_visits": (["lawatan pasukan"], ["lawatan pasukan saya"]),
    "team_activity": (["aktiviti pasukan"], ["peta pasukan saya"]),
    "team_forecast": (["ramalan pasukan"], ["unjuran pasukan"]),
}
MS_LEADS = {
    "train": ["buka {n}", "pergi ke {n}", "tunjukkan {n}", "paparkan {n}", "saya nak tengok {n}", "bawa saya ke {n}", "lihat {n}"],
    "validation": ["jom ke {n}", "buka halaman {n}"],
    "test": ["tolong buka {n}", "tunjuk {n}", "saya mahu ke {n}"],
}

NAV_EXTRA_INTENTS = {
    "all_orders": ["orders from customers waiting for me", "the customer orders i need to approve",
                   "sales orders to approve", "let me see the orders my reps sent in",
                   "which customer orders came in", "approve the orders from shops",
                   "retailer orders that need a decision", "orders placed by the shops"],
    "purchase_order": ["orders we place with suppliers", "what we ordered from the supplier",
                       "supplier purchase orders", "stock we are buying in",
                       "purchase orders issued to suppliers", "the po for the vendor"],
}
NAV_EXTRA_NOUNS = {"all_orders": ["incoming orders", "order queue", "retailer orders"]}

IMPORT_FRAMES = {
    "train": ["import a csv file", "upload a csv", "i want to upload a spreadsheet", "load a csv", "bring in a csv",
              "import products from csv", "muat naik fail csv", "import data dari csv"],
    "validation": ["upload an excel sheet", "import my spreadsheet", "muat naik excel"],
    "test": ["i need to import a csv", "put a csv into the system", "import fail csv", "upload csv file please"],
}
CONNECT = {
    "train": ["{a} and then {b}", "{a} then {b}", "{a} and also {b}", "{a} followed by {b}", "first {a} then {b}"],
    "validation": ["{a} and {b} together", "{a} after that {b}"],
    "test": ["{a}, then {b}", "{a} plus {b}", "{a} and next {b}"],
}

ACTION_FRAMES = {
    "search_orders": SEARCH, "daily_briefing": BRIEFING, "export_results": EXPORT,
    "explain_item": EXPLAIN, "set_item_status": SET_STATUS, "brief_retailer": BRIEF,
}
MIX = [("nav", 0.28), ("nav_ms", 0.05), ("search_orders", 0.17), ("daily_briefing", 0.05), ("export_results", 0.05),
       ("explain_item", 0.07), ("set_item_status", 0.07), ("brief_retailer", 0.07),
       ("write_negative", 0.04), ("compound_action", 0.04), ("import_contrast", 0.03), ("gated", 0.08)]
_ORIGINAL_NAV = (copy.deepcopy(md.INTENTS), copy.deepcopy(md.NOUNS))
_PLACEHOLDER = re.compile(r"\{(sp|rt|pd|st|sv|ord|n|no)\}")


def norm(text: str) -> str:
    return " ".join(text.lower().split())


def install_navigation_source() -> None:
    intents, nouns = _ORIGINAL_NAV
    md.INTENTS = {k: [s for s in v if s not in NAV_INTENT_BLOCKLIST] + NAV_EXTRA_INTENTS.get(k, [])
                  for k, v in intents.items()}
    md.NOUNS = {k: [s for s in v if s not in NAV_NOUN_BLOCKLIST] + NAV_EXTRA_NOUNS.get(k, [])
                for k, v in nouns.items()}


def _typo_chunk(chunk: str, r: random.Random, p: float) -> str:
    if r.random() >= p:
        return chunk
    tokens = re.split(r"(\s+)", chunk)
    words = [i for i, t in enumerate(tokens) if t.strip() and len(t) >= 5]
    if not words:
        return chunk
    i = r.choice(words)
    tokens[i] = md.typo(tokens[i], r)
    return "".join(tokens)


def _surface(name: str, r: random.Random) -> str:
    return name.lower() if r.random() < 0.35 else name


def _phrase(table: dict, lang: str, key: str, group: str, r: random.Random) -> str:
    language = r.choice(["en", "ms"]) if lang == "mix" else lang
    return r.choice(table[language][key][group])


def _group(split: str) -> str:
    return "train" if split == "train" else "held"


def _fill(frame: str, lang: str, split: str, r: random.Random, args: dict[str, Any]) -> str:
    group = _group(split)
    position = r.randint(1, 10)
    out: list[str] = []
    last = 0
    for match in _PLACEHOLDER.finditer(frame):
        out.append(_typo_chunk(frame[last:match.start()], r, 0.06))
        slot = match.group(1)
        if slot == "sp":
            value = _surface(r.choice(SALESPEOPLE[split]), r)
            args["salesperson"] = value
            out.append(value)
        elif slot == "rt":
            value = _surface(r.choice(RETAILERS[split]), r)
            args["retailer"] = value
            out.append(value)
        elif slot == "pd":
            key = r.choice(sorted(PERIOD["en"]))
            args["period"] = key
            out.append(_phrase(PERIOD, lang, key, group, r))
        elif slot == "st":
            key = r.choice(sorted(ORDER_STATUS["en"]))
            args["status"] = key
            out.append(_phrase(ORDER_STATUS, lang, key, group, r))
        elif slot == "sv":
            key = r.choice(sorted(ITEM_STATUS["en"]))
            args["status"] = key
            out.append(_phrase(ITEM_STATUS, lang, key, group, r))
        elif slot == "no":
            out.append(str(r.randint(1000, 9999)))
        elif slot == "ord":
            args["position"] = position
            language = r.choice(["en", "ms"]) if lang == "mix" else lang
            out.append(ORDINAL[language][position - 1])
        else:
            args["position"] = position
            out.append(str(position))
        last = match.end()
    out.append(_typo_chunk(frame[last:], r, 0.06))
    return "".join(out)


def _decorate(text: str, r: random.Random, split: str = "train") -> str:
    if r.random() < 0.15:
        text = f"{r.choice(POLITE[split])} {text}"
    elif r.random() < 0.08:
        text = f"{r.choice(md.FILLERS)} {text}"
    x = r.random()
    if x < 0.05:
        text += "."
    elif x < 0.07:
        text += "?"
    return " ".join(text.split())


def pick_actions(r: random.Random, must: str | None = None, exclude: str | None = None) -> list[str]:
    names = [a for a in schema.ACTION_ORDER if a != exclude]
    if r.random() < 0.7:
        chosen = set(names)
    else:
        chosen = set(r.sample(names, r.randint(0 if must is None else 1, max(1, len(names) - 1))))
        if must:
            chosen.add(must)
    return [a for a in schema.ACTION_ORDER if a in chosen]


def _row(text: str, pages: list[str], actions: list[str], target: dict, kind: str, family: str, lang: str,
         r: random.Random) -> dict:
    return {
        "text": " ".join(text.split()),
        "pages": pages,
        "aliasSeed": 0 if r.random() < 0.6 else r.randint(1, 9999),
        "actions": actions,
        "target": target,
        "kind": kind,
        "family": family,
        "lang": lang,
    }


def gen_action_row(action: str, split: str, r: random.Random, gated: bool = False) -> dict:
    frame_entry = r.choice(ACTION_FRAMES[action][split])
    lang, frame = frame_entry[0], frame_entry[1]
    args: dict[str, Any] = {}
    text = _fill(frame, lang, split, r, args)
    args.update(frame_entry[2] if len(frame_entry) > 2 else {})
    pages = md.pick_pages(r, None, True)
    if gated:
        actions = pick_actions(r, exclude=action)
        target = {"name": schema.NO_MATCH, "args": {}}
        kind = "gated"
    else:
        actions = pick_actions(r, must=action)
        target = {"name": action, "args": args}
        kind = action
    return _row(_decorate(text, r, split), pages, actions, target, kind, f"{action}:{frame}", lang, r)


def gen_write_negative(split: str, r: random.Random) -> dict:
    frame = r.choice(WRITE_NEGATIVE[split])
    text = _fill(frame, "en", split, r, {})
    pages = md.pick_pages(r, None, True)
    lang = "ms" if re.search(r"\b(luluskan|tolak|padam|simpan|sediakan|terbitkan|batalkan)\b", frame) else "en"
    return _row(_decorate(text, r, split), pages, pick_actions(r), {"name": schema.NO_MATCH, "args": {}},
                "write_negative", f"write:{frame}", lang, r)


def gen_nav_ms_row(split: str, r: random.Random) -> dict:
    key = r.choice(sorted(MS_NOUNS))
    base, held = MS_NOUNS[key]
    noun = r.choice(base if split == "train" else base + held)
    lead = r.choice(MS_LEADS[split])
    absent = r.random() < 0.15
    pages = md.pick_pages(r, key, not absent)
    target = (
        {"name": schema.NO_MATCH, "args": {}} if absent else {"name": schema.NAVIGATE, "args": {"page": key}}
    )
    text = _typo_chunk(lead.split("{n}")[0], r, 0.05) + noun + lead.split("{n}")[1]
    return _row(_decorate(text, r, split), pages, pick_actions(r), target, "nav_ms", f"navms:{lead}", "ms", r)


def gen_compound_action(split: str, r: random.Random) -> dict:
    parts = []
    for action in r.sample(sorted(ACTION_FRAMES), 2):
        entry = r.choice(ACTION_FRAMES[action][split])
        parts.append(_fill(entry[1], entry[0], split, r, {}))
    connector = r.choice(CONNECT[split])
    text = connector.format(a=parts[0], b=parts[1])
    return _row(_decorate(text, r, split), md.pick_pages(r, None, True), pick_actions(r),
                {"name": schema.NO_MATCH, "args": {}}, "compound_action", f"compound:{connector}", "en", r)


def gen_import_contrast(split: str, r: random.Random) -> dict:
    phrase = r.choice(IMPORT_FRAMES[split])
    present = r.random() < 0.4
    pages = md.pick_pages(r, "import_preview", present)
    target = (
        {"name": schema.NAVIGATE, "args": {"page": "import_preview"}} if present
        else {"name": schema.NO_MATCH, "args": {}}
    )
    actions = pick_actions(r, must="export_results")
    return _row(_decorate(phrase, r, split), pages, actions, target, "import_contrast", f"import:{phrase}",
                "ms" if phrase.startswith(("muat", "import data", "import fail")) else "en", r)


def gen_nav_row(split: str, r: random.Random) -> dict:
    kinds = [k for k, _ in md.MIX]
    old = md.gen_row(r.choices(kinds, [w for _, w in md.MIX])[0], split, r)
    target = (
        {"name": schema.NO_MATCH, "args": {}}
        if old["target"] == md.NO_MATCH
        else {"name": schema.NAVIGATE, "args": {"page": old["target"]}}
    )
    return {
        "text": old["text"], "pages": old["pages"], "aliasSeed": old["aliasSeed"],
        "actions": pick_actions(r), "target": target,
        "kind": "nav:" + old["kind"], "family": old["family"], "lang": "en",
    }


def gen_row(kind: str, split: str, r: random.Random) -> dict:
    if kind == "nav":
        return gen_nav_row(split, r)
    if kind == "write_negative":
        return gen_write_negative(split, r)
    if kind == "nav_ms":
        return gen_nav_ms_row(split, r)
    if kind == "compound_action":
        return gen_compound_action(split, r)
    if kind == "import_contrast":
        return gen_import_contrast(split, r)
    if kind == "gated":
        return gen_action_row(r.choice(sorted(ACTION_FRAMES)), split, r, gated=True)
    return gen_action_row(kind, split, r)


def to_request(row: dict) -> dict:
    return {"text": row["text"], "pages": md.request_pages(row), "actions": row["actions"]}


def build(split: str, n: int, seed: int, banned: set[str]) -> list[dict]:
    r = random.Random(f"{seed}:{split}")
    kinds = [k for k, _ in MIX]
    weights = [w for _, w in MIX]
    out: list[dict] = []
    seen: set[tuple[str, tuple[str, ...], tuple[str, ...]]] = set()
    guard = 0
    while len(out) < n and guard < n * 80:
        guard += 1
        row = gen_row(r.choices(kinds, weights)[0], split, r)
        sig = (norm(row["text"]), tuple(row["pages"]), tuple(row["actions"]))
        if norm(row["text"]) in banned or sig in seen:
            continue
        seen.add(sig)
        out.append(row)
    return out


def verbatim_errors(rows: list[dict]) -> list[str]:
    bad = []
    for row in rows:
        for key in ("salesperson", "retailer"):
            value = row["target"]["args"].get(key)
            if value is not None and value not in row["text"]:
                bad.append(f"{key}={value!r} not in {row['text']!r}")
    return bad


def conflicts(*splits: list[dict]) -> list[str]:
    seen: dict[str, str] = {}
    bad = []
    for rows in splits:
        for row in rows:
            if row["target"]["name"] == schema.NO_MATCH:
                continue
            key = norm(row["text"])
            call = schema.serialize_call(row["target"]).lower()
            if seen.setdefault(key, call) != call:
                bad.append(f"{row['text']!r}: {seen[key]} vs {call}")
    return bad


def frozen_texts() -> set[str]:
    texts: set[str] = set()
    for name in FROZEN_FILES:
        path = LLM_DIR / name
        if path.exists():
            texts |= {norm(c["text"]) for c in json.loads(path.read_text(encoding="utf-8"))["cases"]}
    return texts


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out-dir", type=Path, default=DATA)
    args = ap.parse_args(argv)
    install_navigation_source()
    frozen = frozen_texts()
    test = build("test", SIZES["test"], SEED, frozen)
    test_texts = {norm(x["text"]) for x in test}
    val = build("validation", SIZES["validation"], SEED, frozen | test_texts)
    val_texts = {norm(x["text"]) for x in val}
    train = build("train", SIZES["train"], SEED, frozen | test_texts | val_texts)
    train_texts = {norm(x["text"]) for x in train}
    for name, held in (("frozen evaluation files", frozen), ("validation", val_texts), ("test", test_texts)):
        overlap = train_texts & held
        if overlap:
            sys.exit(f"train overlaps {name}: {sorted(overlap)[:5]}")
    if test_texts & val_texts or test_texts & frozen or val_texts & frozen:
        sys.exit("held-out sets overlap")
    errors = verbatim_errors(train + val + test) + conflicts(train, val, test)
    if errors:
        sys.exit("dataset invariants failed:\n" + "\n".join(errors[:10]))
    args.out_dir.mkdir(parents=True, exist_ok=True)
    for name, rows in (("train", train), ("validation", val), ("test", test)):
        path = args.out_dir / f"{name}.jsonl"
        with path.open("w", encoding="utf-8", newline="\n") as handle:
            handle.write("".join(json.dumps(x, ensure_ascii=False) + "\n" for x in rows))
        kinds: dict[str, int] = {}
        for x in rows:
            group = x["kind"].split(":")[0]
            kinds[group] = kinds.get(group, 0) + 1
        langs: dict[str, int] = {}
        for x in rows:
            langs[x["lang"]] = langs.get(x["lang"], 0) + 1
        digest = hashlib.sha256(path.read_bytes()).hexdigest()[:16]
        print(f"{name}: {len(rows)} rows, no_match={sum(x['target']['name'] == schema.NO_MATCH for x in rows)}, "
              f"sha256[:16]={digest}, kinds={kinds}, lang={langs}")
    print("OK: no train text equals a validation, test or frozen-evaluation text; names copied verbatim")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/train/train_lora.py
from __future__ import annotations

import argparse
import hashlib
import json
import math
import os
import random
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer

from tools.voice.dashboard_llm import service
from tools.voice.dashboard_llm.train.make_dataset import DATA, request_pages

HERE = Path(__file__).resolve().parent
BASE_DIR = os.environ.get("DASHBOARD_VOICE_MODEL_DIR", service.DEFAULT_MODEL_DIR)
ADAPTER_OUT = os.environ.get("DASHBOARD_VOICE_ADAPTER_DIR", service.DEFAULT_ADAPTER_DIR)
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])


def sha(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


def encode(tok, rows: list[dict]) -> list[tuple[list[int], int]]:
    out = []
    for r in rows:
        prompt = service.render_prompt(tok, r["text"], request_pages(r))
        p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        t_ids = tok(f"{r['target']}<escape>", add_special_tokens=False)["input_ids"]
        assert t_ids[-1] == service.ESCAPE_ID, "target must end with <escape>"
        out.append((p_ids + t_ids, len(t_ids)))
    return out


def batches(items, size, shuffle_rng=None):
    idx = list(range(len(items)))
    if shuffle_rng:
        shuffle_rng.shuffle(idx)
        chunks = [sorted(idx[i : i + size * 16], key=lambda j: len(items[j][0])) for i in range(0, len(idx), size * 16)]
        bs = [c[i : i + size] for c in chunks for i in range(0, len(c), size)]
        shuffle_rng.shuffle(bs)
    else:
        bs = [idx[i : i + size] for i in range(0, len(idx), size)]
    return bs


def loss_for(peft_model, items, batch, device, pad_id):
    base = peft_model.get_base_model()
    width = max(len(items[j][0]) for j in batch)
    ids = torch.full((len(batch), width), pad_id, dtype=torch.long)
    mask = torch.zeros((len(batch), width), dtype=torch.long)
    pos, tgt = [], []
    for b, j in enumerate(batch):
        seq, k = items[j]
        ids[b, : len(seq)] = torch.tensor(seq)
        mask[b, : len(seq)] = 1
        for t in range(len(seq) - k, len(seq)):
            pos.append((b, t - 1))
            tgt.append(seq[t])
    hidden = base.model(input_ids=ids.to(device), attention_mask=mask.to(device)).last_hidden_state
    sel = hidden[[p[0] for p in pos], [p[1] for p in pos]]
    logits = base.lm_head(sel).float()
    return F.cross_entropy(logits, torch.tensor(tgt, device=device))


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--micro-batch", type=int, default=2)
    ap.add_argument("--accum", type=int, default=8)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--eval-every", type=int, default=50, help="optimizer steps")
    ap.add_argument("--patience", type=int, default=3)
    ap.add_argument("--max-steps", type=int, default=0, help="debug cap")
    ap.add_argument("--seed", type=int, default=20260930)
    ap.add_argument("--data-dir", type=Path, default=DATA)
    ap.add_argument("--metadata-out", type=Path, default=HERE / "RUN_METADATA.json")
    args = ap.parse_args()

    random.seed(args.seed)
    torch.manual_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(BASE_DIR, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(BASE_DIR, local_files_only=True, dtype=torch.float32)
    from peft import LoraConfig, get_peft_model

    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    peft_model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", **LORA)).to(device)
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)

    def load(name):
        return [json.loads(line) for line in (args.data_dir / f"{name}.jsonl").read_text(encoding="utf-8").splitlines()]

    train = encode(tok, load("train"))
    val = encode(tok, load("validation"))
    lens = sorted(len(x[0]) for x in train)
    print(f"train {len(train)} val {len(val)} tokens p50 {lens[len(lens)//2]} max {lens[-1]} trainable {trainable}", flush=True)

    opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=args.lr, weight_decay=0.0)
    steps_per_epoch = math.ceil(len(train) / (args.micro_batch * args.accum))
    total = steps_per_epoch * args.epochs
    warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    )

    def evaluate() -> float:
        peft_model.eval()
        tot, n = 0.0, 0
        with torch.no_grad():
            for b in batches(val, 4):
                tot += float(loss_for(peft_model, val, b, device, tok.pad_token_id)) * len(b)
                n += len(b)
        peft_model.train()
        return tot / n

    rng = random.Random(args.seed)
    peft_model.train()
    curve, best, bad, step, started = [], float("inf"), 0, 0, time.time()
    stop = False
    Path(ADAPTER_OUT).mkdir(parents=True, exist_ok=True)
    for epoch in range(args.epochs):
        run, cnt = 0.0, 0
        micro = batches(train, args.micro_batch, rng)
        for i, b in enumerate(micro):
            loss = loss_for(peft_model, train, b, device, tok.pad_token_id)
            (loss / args.accum).backward()
            run += float(loss)
            cnt += 1
            if (i + 1) % args.accum and i + 1 != len(micro):
                continue
            torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 1.0)
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            if step % args.eval_every == 0 or step == total or (args.max_steps and step >= args.max_steps):
                v = evaluate()
                curve.append({"step": step, "epoch": round(epoch + (i + 1) / len(micro), 2), "trainLoss": round(run / cnt, 5), "valLoss": round(v, 5)})
                print(curve[-1], f"{time.time()-started:.0f}s vram {torch.cuda.max_memory_allocated()/2**20:.0f}MiB" if device == "cuda" else "", flush=True)
                run, cnt = 0.0, 0
                if v < best - 1e-4:
                    best, bad = v, 0
                    peft_model.save_pretrained(ADAPTER_OUT)
                else:
                    bad += 1
                    if bad >= args.patience:
                        print("early stopping", flush=True)
                        stop = True
            if stop or (args.max_steps and step >= args.max_steps):
                stop = True
                break
        if stop:
            break

    import peft
    import transformers

    meta = {
        "base": "functiongemma-270m-it",
        "adapterDir": str(Path(ADAPTER_OUT).expanduser().resolve()),
        "lora": LORA,
        "hyperparameters": {"epochs": args.epochs, "microBatch": args.micro_batch, "gradAccum": args.accum, "lr": args.lr, "schedule": "cosine, 5% warmup", "seed": args.seed, "precision": "fp32", "gradientCheckpointing": True},
        "data": {n: {"rows": len(load(n)), "sha256": sha(args.data_dir / f"{n}.jsonl")} for n in ("train", "validation", "test")},
        "makeDatasetSha256": sha(HERE / "make_dataset.py"),
        "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__},
        "device": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
        "trainableParams": trainable,
        "optimizerSteps": step,
        "wallSeconds": round(time.time() - started),
        "peakVramMiB": round(torch.cuda.max_memory_allocated() / 2**20) if device == "cuda" else None,
        "bestValLoss": round(best, 5),
        "curve": curve,
    }
    args.metadata_out.parent.mkdir(parents=True, exist_ok=True)
    args.metadata_out.write_text(json.dumps(meta, indent=1), encoding="utf-8")
    print("done; best val loss", meta["bestValLoss"], "adapter", ADAPTER_OUT)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /kaggle/working/fyp/tools/voice/dashboard_llm/train/train_copilot_lora.py
from __future__ import annotations

import argparse
import json
import math
import random
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from tools.voice.dashboard_llm import copilot_schema as schema
from tools.voice.dashboard_llm.train import make_copilot_dataset as ds
from tools.voice.dashboard_llm.train.train_lora import LORA, batches, loss_for, sha

HERE = Path(__file__).resolve().parent
END_CALL_ID = 49


def encode(tok, rows: list[dict]) -> list[tuple[list[int], int]]:
    out = []
    for row in rows:
        request = ds.to_request(row)
        prompt = schema.render_prompt(tok, request["text"], request["pages"], request["actions"])
        p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
        t_ids = tok(schema.serialize_call(row["target"]), add_special_tokens=False)["input_ids"]
        assert t_ids[-1] == END_CALL_ID, "target must end with <end_function_call>"
        out.append((p_ids + t_ids, len(t_ids)))
    return out


def main(argv: list[str] | None = None) -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--base-dir", required=True, help="local FunctionGemma-270M-it folder")
    ap.add_argument("--adapter-out", type=Path, required=True)
    ap.add_argument("--data-dir", type=Path, default=ds.DATA)
    ap.add_argument("--metadata-out", type=Path)
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--micro-batch", type=int, default=2)
    ap.add_argument("--accum", type=int, default=8)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--eval-every", type=int, default=50, help="optimizer steps")
    ap.add_argument("--patience", type=int, default=3,
                    help="stop after this many checks without a better validation loss; 0 = never stop early "
                         "(the first run stopped at step 700 of 1500, before the learning rate had decayed)")
    ap.add_argument("--save-final", type=Path, help="also save the adapter after the last step to this folder")
    ap.add_argument("--val-limit", type=int, default=0, help="check only the first N validation rows (faster)")
    ap.add_argument("--no-grad-checkpoint", action="store_true",
                    help="faster, uses more GPU memory; fine on a 16 GB GPU, not on a 4 GB one")
    ap.add_argument("--max-steps", type=int, default=0, help="debug cap")
    ap.add_argument("--limit", type=int, default=0, help="use only the first N training rows (debug)")
    ap.add_argument("--seed", type=int, default=20261006)
    args = ap.parse_args(argv)
    metadata_out = args.metadata_out or args.adapter_out / "RUN_METADATA.json"

    random.seed(args.seed)
    torch.manual_seed(args.seed)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tok = AutoTokenizer.from_pretrained(args.base_dir, local_files_only=True)
    if tok.convert_tokens_to_ids(schema.END_CALL) != END_CALL_ID:
        raise SystemExit("unexpected tokenizer: <end_function_call> id changed")
    model = AutoModelForCausalLM.from_pretrained(args.base_dir, local_files_only=True, dtype=torch.float32)
    from peft import LoraConfig, get_peft_model

    if not args.no_grad_checkpoint:
        model.gradient_checkpointing_enable()
        model.enable_input_require_grads()
    peft_model = get_peft_model(model, LoraConfig(task_type="CAUSAL_LM", **LORA)).to(device)
    trainable = sum(p.numel() for p in peft_model.parameters() if p.requires_grad)

    def load(name: str) -> list[dict]:
        return [json.loads(x) for x in (args.data_dir / f"{name}.jsonl").read_text(encoding="utf-8").splitlines()]

    train_rows = load("train")[: args.limit or None]
    train, val = encode(tok, train_rows), encode(tok, load("validation")[: args.val_limit or None])
    lens = sorted(len(x[0]) for x in train)
    print(f"train {len(train)} val {len(val)} tokens p50 {lens[len(lens) // 2]} max {lens[-1]} trainable {trainable}", flush=True)

    opt = torch.optim.AdamW([p for p in peft_model.parameters() if p.requires_grad], lr=args.lr, weight_decay=0.0)
    total = math.ceil(len(train) / (args.micro_batch * args.accum)) * args.epochs
    warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm)))
    )

    def evaluate() -> float:
        peft_model.eval()
        tot, n = 0.0, 0
        with torch.no_grad():
            for b in batches(val, 4):
                tot += float(loss_for(peft_model, val, b, device, tok.pad_token_id)) * len(b)
                n += len(b)
        peft_model.train()
        return tot / n

    rng = random.Random(args.seed)
    peft_model.train()
    curve, best, bad, step, started = [], float("inf"), 0, 0, time.time()
    stop = False
    args.adapter_out.mkdir(parents=True, exist_ok=True)
    for epoch in range(args.epochs):
        run, cnt = 0.0, 0
        micro = batches(train, args.micro_batch, rng)
        for i, b in enumerate(micro):
            loss = loss_for(peft_model, train, b, device, tok.pad_token_id)
            (loss / args.accum).backward()
            run += float(loss)
            cnt += 1
            if (i + 1) % args.accum and i + 1 != len(micro):
                continue
            torch.nn.utils.clip_grad_norm_(peft_model.parameters(), 1.0)
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)
            step += 1
            if step % args.eval_every == 0 or step == total or (args.max_steps and step >= args.max_steps):
                v = evaluate()
                curve.append({"step": step, "epoch": round(epoch + (i + 1) / len(micro), 2),
                              "trainLoss": round(run / cnt, 5), "valLoss": round(v, 5)})
                vram = f" vram {torch.cuda.max_memory_allocated() / 2**20:.0f}MiB" if device == "cuda" else ""
                print(curve[-1], f"{time.time() - started:.0f}s{vram}", flush=True)
                run, cnt = 0.0, 0
                if v < best - 1e-4:
                    best, bad = v, 0
                    peft_model.save_pretrained(args.adapter_out)
                else:
                    bad += 1
                    if args.patience and bad >= args.patience:
                        print("early stopping", flush=True)
                        stop = True
            if stop or (args.max_steps and step >= args.max_steps):
                stop = True
                break
        if stop:
            break

    if args.save_final:
        args.save_final.mkdir(parents=True, exist_ok=True)
        peft_model.save_pretrained(args.save_final)
        print("final adapter saved to", args.save_final, flush=True)

    import peft
    import transformers

    meta = {
        "base": "functiongemma-270m-it",
        "adapterDir": str(args.adapter_out.expanduser().resolve()),
        "lora": LORA,
        "hyperparameters": {"epochs": args.epochs, "microBatch": args.micro_batch, "gradAccum": args.accum,
                            "lr": args.lr, "schedule": "cosine, 5% warmup", "seed": args.seed, "precision": "fp32",
                            "gradientCheckpointing": not args.no_grad_checkpoint, "trainRows": len(train),
                            "patience": args.patience, "valRows": len(val)},
        "data": {n: {"rows": len(load(n)), "sha256": sha(args.data_dir / f"{n}.jsonl")} for n in ("train", "validation", "test")},
        "makeCopilotDatasetSha256": sha(HERE / "make_copilot_dataset.py"),
        "copilotSchemaSha256": sha(HERE.parent / "copilot_schema.py"),
        "versions": {"torch": torch.__version__, "transformers": transformers.__version__, "peft": peft.__version__},
        "device": torch.cuda.get_device_name(0) if device == "cuda" else "cpu",
        "trainableParams": trainable,
        "optimizerSteps": step,
        "wallSeconds": round(time.time() - started),
        "peakVramMiB": round(torch.cuda.max_memory_allocated() / 2**20) if device == "cuda" else None,
        "bestValLoss": round(best, 5),
        "curve": curve,
    }
    metadata_out.parent.mkdir(parents=True, exist_ok=True)
    metadata_out.write_text(json.dumps(meta, indent=1), encoding="utf-8")
    print("done; best val loss", meta["bestValLoss"], "adapter", args.adapter_out)


if __name__ == "__main__":
    main()


In [ ]:
import hashlib, json
EXPECTED = {
 "tools/voice/dashboard_llm/__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
 "tools/voice/dashboard_llm/service.py": "72378cd2a29fbebdf9820fe50b9d42827698737d2763682a4237c326b5ac5312",
 "tools/voice/dashboard_llm/copilot_schema.py": "018cbe6c1ed009907c299675e911f56432a17a3ec7484bc7763d3a6c9ff87e9f",
 "tools/voice/dashboard_llm/copilot_model.py": "5a8832ce27e384832003f91f1220a0e20f761f815c6da827372a334dfd3904f0",
 "tools/voice/dashboard_llm/evaluate_copilot.py": "955d397f95150d4c10daf8c7ee6648bde3f24cf6522e8afdc303d6a83bc46aa4",
 "tools/voice/dashboard_llm/eval_phrases.json": "37f5dfb6b0037b10b6c313e84dc0b494b5619dcc9f2ed853f1b7be50d8bfba82",
 "tools/voice/dashboard_llm/fresh_phrases.json": "e3209ebeeda01e6c924a9469994c765571aff3790427987021a2331a312f7132",
 "tools/voice/dashboard_llm/copilot_eval_phrases.json": "962d3f50b84f8b280274eaaf19dbfefa1162128470df09c15384c4bcd5780dd8",
 "tools/voice/dashboard_llm/train/make_dataset.py": "d220ce5231e1ff6dcaf78600a657be6ff8ad2feaf9395e9b297a57f711d856a3",
 "tools/voice/dashboard_llm/train/make_copilot_dataset.py": "75390c72b33356b6549060b21f5afcf925d73b0215e7ed279882d92c1bafe0d8",
 "tools/voice/dashboard_llm/train/train_lora.py": "b3e938efe4b440eb193a434cf9815dafb6ed66291de93c09e84c61cc1d544312",
 "tools/voice/dashboard_llm/train/train_copilot_lora.py": "7d249e96a901d07d679faea6b8fd4f8a51293b03215ded1b5d8d00f9f9467585"
}
for rel, want in EXPECTED.items():
    text = open('/kaggle/working/fyp/' + rel, encoding='utf-8').read()
    got = hashlib.sha256(text.encode('utf-8')).hexdigest()
    assert got == want, 'file differs from the repository: ' + rel
print('all', len(EXPECTED), 'source files match')

## Dataset

The generator is deterministic. The hashes must match the ones computed on the repository machine.

In [ ]:
!cd /kaggle/working/fyp && python -m tools.voice.dashboard_llm.train.make_copilot_dataset --out-dir /kaggle/working/data
WANT = {
 "train": "ab0b53fecc07b0b98227c4a23b7e2b783e335d5ba5b0707511813806f53d1cbd",
 "validation": "1094b602df5ef48c9ea8f06ad33bff61ee1a241e38854d9e64d1282536343129",
 "test": "908233d9dcfae50794d1c8d46786a51f4af6c9a88450d577c3f6df1aa884263b"
}
same = True
for name, want in WANT.items():
    got = hashlib.sha256(open('/kaggle/working/data/' + name + '.jsonl', 'rb').read()).hexdigest()
    if got != want:
        same = False
        print('WARNING: dataset', name, 'differs from the repository build:', got[:16], 'instead of', want[:16])
print('dataset matches the repository build' if same else
      'The dataset differs (a different Python version can cause this). The generator has already checked '
      'that no training sentence equals a validation, test or frozen sentence. Tell the repository owner.')

## Train

LoRA (r=16) on the 270M model, 3 epochs, effective batch 16. The full schedule runs (no early stop) and the final adapter is kept. If you see an out-of-memory error, change `--micro-batch 8 --accum 2` to `--micro-batch 4 --accum 4` and run the cell again.

In [ ]:
!cd /kaggle/working/fyp && PYTHONPATH=/kaggle/working/fyp python -m tools.voice.dashboard_llm.train.train_copilot_lora --base-dir /kaggle/working/base --adapter-out /kaggle/working/adapter --data-dir /kaggle/working/data --micro-batch 8 --accum 2 --epochs 3 --eval-every 250 --val-limit 400 --patience 0 --no-grad-checkpoint --save-final /kaggle/working/adapter_final

## Evaluate

Frozen hand-written set, generated test split and the 80 older navigation phrases, for the untuned base model and the adapter.

In [ ]:
!cd /kaggle/working/fyp && PYTHONPATH=/kaggle/working/fyp python -m tools.voice.dashboard_llm.evaluate_copilot --model-dir /kaggle/working/base --adapter-dir /kaggle/working/adapter_final --data-dir /kaggle/working/data --output /kaggle/working/report.json

In [ ]:
report = json.load(open('/kaggle/working/report.json', encoding='utf-8'))
for key, result in report['results'].items():
    print(key, result['latencyMs'])
    for row in result['sweep']:
        if row['threshold'] in (0.0, 0.9, 0.95, 0.98, 0.99):
            print('  thr %.2f  wrong %.1f%%  yield %.1f%%  abstain-on-negatives %.1f%%' % (
                row['threshold'], 100*row['wrongCall'], 100*row['yield'], 100*row['abstainOnNegatives']))

## Package the result

Download `copilot_adapter.zip` and `report.json` from the Output panel, then give them to the repository owner.

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/copilot_adapter', 'zip', '/kaggle/working/adapter_final')
print(os.listdir('/kaggle/working'))